# Manuscript Code Reference
This notebook contains all scripts relevant to the "Epigenetic patterns and methylation-based models for robust outcome prediction in osteosarcoma" manuscript
* Section 1: Data Preprocessing
* Section 2: VMR Derivation and Exploration
* Section 3: Univariable Analysis
* Section 4: Supervised Analysis
* Section 5: Validation Cohort Analysis
* Section 6: Epigenetic Clock Analysis
* Section 7: Multi-Omics Comparative Analysis

Any correspondence regarding this code reference should be directed to jjbowers@mgh.harvard.edu (Primary) or corresponding author DSPENTZOS@mgh.harvard.edu (Secondary)

## Section 1: Data Preprocessing

### Script Name: Methylation Preprocessing Pipeline

**Authors**: Christopher Lietz and Joshua Bowers

**Language**: R

**Description**: This script implements a complete pipeline to preprocess Illumina 450k methylation array-based data using "Minfi" package in R. Starting from raw IDAT files, it produces cleaned and normalized methylation matricies in the form of Beta and M- values. More specifically, this pipeline performs quality control, sex predictions, applies functional normalization, and annotates CpGs with functional genomic context. It also remvoes probes that are low quality, cross-reactive, sex-chromosome-associated, and overlapping with common SNPs.

**Requires**: Raw methylation IDAT files (from TARGET), sample metadata (Requires Sample_Name and Basename), and a list of Chen's cross-reactive probes (https://pmc.ncbi.nlm.nih.gov/articles/PMC3592906/)

In [ ]:
##### Dependencies #####
library(data.table)
library(RColorBrewer)
library(GenomicRanges)
library(minfi)
library(readr)

# May need to switch this out for the epic array manifest and annotations if using that array
library('IlluminaHumanMethylation450kmanifest') # BiocManager::install('IlluminaHumanMethylation450kmanifest')
library('IlluminaHumanMethylation450kanno.ilmn12.hg19') # BiocManager::install('IlluminaHumanMethylation450kanno.ilmn12.hg19')


#################### Minfi Preprocessing ####################
# Set working directory as location of raw idat files to be analyzed
setwd("H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_raw/")

# Assuming there is a metharray sample sheet with all of the idat file names under a column called 
# "Basename" and that there is only one csv under this directory.  Sample name should also be included
targets <- read.metharray.sheet("//Volumes/homedir$/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_raw")

# check files
list.files("/H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_raw/")

# read in the methylation data
RGSet <- read.metharray.exp(targets = targets)

# create a methylset of raw meth data for QC
MSet <- preprocessRaw(RGSet) 

# create a ratioset to get methylation values for each CpG
RSet <- ratioConvert(MSet, what = "both", keepCN = TRUE)

# map ratioset to reference genome
GRset <- mapToGenome(RSet)

# get probe ranges
gr <- granges(GRset)

# set accessors (pdatas are the variables to be associated with methylation)
sampleNames <- sampleNames(GRset)
probeNames <- featureNames(GRset)
#pheno <- pData(GRset)



##### Quality Control #####
##QC note that the defult color scheme does not handle > 8 classes
qc <- getQC(MSet)
plotQC(qc)

densityPlot(MSet, sampGroups = targets$batch)
#densityPlot(MSet)
densityBeanPlot(MSet, sampGroups = targets$batch)

# predict and plot sex
predictedSex <- getSex(GRset, cutoff = -2)
GRset <- addSex(GRset, predictedSex)
plotSex(GRset)

# qc report PDF - Will be placed in working directory in current form
qcReport(RGSet, pdf = "qcReport.pdf")



##### Preprocess / Normalize / Annotate the methylation data and drop SNPs #####
# Process data
GRset.funnorm <- preprocessFunnorm(RGSet)

# Add resort type context
type <- getIslandStatus(GRset.funnorm)
mcols(GRset.funnorm@rowRanges) <- cbind(mcols(granges(GRset.funnorm)), type)

# Drop CpGs based on detection p val and plot detection p vals to id failed samples
detP <- detectionP(RGSet)
#detP <- detP[match(featureNames(RGSet),rownames(GRset.funnorm)),] 
pal <- brewer.pal(8,"Dark2")
barplot(colMeans(detP), las=2, col=pal[factor(targets$batch)],
        cex.names=0.8, ylab="Mean detection p-values")

# Fail in x percent of samples
# Set failure criteria
failed <- detP > 0.01

# Determine fraction of failed site per samp
colMeans(failed)

# How many positions fail in x% of samples - from https://github.com/sirselim/illumina450k_filtering
sum(rowMeans(failed)>0.1)
failed_probes <- rownames(detP[rowMeans(failed)>0.1,])
table(rownames(GRset.funnorm) %in% failed_probes)


# Chen cross react probes - take out of directory after reading in
cross_react <- read.csv("H:/Projects/Methylation_Project/TARGET_Reproducibility/Chen_cross_react_probes.csv", head = T, as.is = T)
cross_react_probes <- as.character(cross_react$TargetID)

# Remove sex chromosomes
ann450k <- getAnnotation(IlluminaHumanMethylation450kanno.ilmn12.hg19)
sex_chr <- ann450k$Name[ann450k$chr %in% c("chrX", "chrY")]

##combine failed, cross react, and sex chr probes - remove dups
#filter.probes <- unique(c(failed.probes, cross.react.probes))
filter_probes <- unique(c(failed_probes, sex_chr, cross_react_probes))
#table(rownames(GRset.funnorm)) %in% filter.probes
to_remove <- rownames(GRset.funnorm) %in% filter_probes
GRset.funnorm_filt <- GRset.funnorm[!to_remove,]

##common SNPs in target site or extension
snps <- getSnpInfo(GRset.funnorm_filt)
GRset.funnorm_filt <- addSnpInfo(GRset.funnorm_filt)
GRset.funnorm_filt <- dropLociWithSnps(GRset.funnorm_filt, snps=c("SBE","CpG"), maf=0)



##### Retrieve Values #####
#get beta values
beta <- getBeta(GRset.funnorm_filt)
colnames(beta) <- targets$Sample_Group[targets$Sample_Name == colnames(beta)]
fwrite(x = as.data.frame(beta), file = "TARGET_funnorm_beta_filt.csv", row.names = T)	

#get m values
Mval <- getM(GRset.funnorm_filt)
colnames(Mval) <- targets$Sample_Group[targets$Sample_Name == colnames(Mval)]
fwrite(x = as.data.frame(Mval), file = "TARGET_funnorm_Mval_filt.csv", row.names = T)

#get copy number
CN <- getCN(GRset.funnorm_filt)
colnames(CN) <- targets$Sample_Group[targets$Sample_Name == colnames(CN)]
fwrite(x = as.data.frame(CN), file = "TARGET_CN_filt.csv", row.names = T)

### Script Name: RNA-Seq Processing Pipeline

**Authors**: Joshua Bowers

**Language**: R and Python

**Description**: These two code blocks take raw per-sample RNA-seq count TSVs and turn them into a normalized expression matrix ready for downstream analyses. The Python part of this script reads in each sample's file based on an excel sample sheet and then 

**Requires**: Raw RNA-seq data (from TARGET) including unstranded reads column and sample metadata sheet (Requires file_name and sample_name)

In [ ]:
import pandas as pd
import os

# Define the directories
raw_data_dir = "H:/Projects/Integrative_Clustering/RNAseq/raw/"
sample_sheet_path = "H:/Projects/Integrative_Clustering/RNAseq/RNAseq_sample_sheet.xlsx"

# Load the sample sheet
sample_sheet = pd.read_excel(sample_sheet_path)

# Initialize an empty dataframe to hold the merged data
merged_df = pd.DataFrame()

# Flag to indicate if 'gene_name' and 'gene_type' have been added
metadata_added = False

# Iterate through the sample sheet to get file_name and sample_name
for _, row in sample_sheet.iterrows():
    file_name = row['file_name']
    sample_name = row['sample_name']

    # Construct the full file path
    file_path = os.path.join(raw_data_dir, file_name)

    # Check if the file exists
    if os.path.exists(file_path):
        # Read the TSV file
        df = pd.read_csv(
            file_path,
            sep='\t',       # Tab-separated values
            skiprows=1,     # Skip the comment line
            header=0,       # Read the second row as the header
            index_col='gene_id'  # Set gene_id as the index
        )

        # Drop the first 4 rows (e.g., N_unmapped, etc.)
        df = df.iloc[4:, :]

        # Extract the 'unstranded' column and rename it to the sample name
        sample_df = df[['unstranded']].rename(columns={'unstranded': sample_name})

        # Add 'gene_name' and 'gene_type' metadata if not already added
        if not metadata_added:
            merged_df[['gene_name', 'gene_type']] = df[['gene_name', 'gene_type']]
            metadata_added = True

        # Merge the current sample data into the main dataframe
        merged_df = merged_df.join(sample_df, how='outer')

# Optionally, save the merged dataframe
# output_path = "H:/Projects/Integrative_Clustering/RNAseq/merged_expression_values_with_metadata_unstranded-counts.csv"
# merged_df.to_csv(output_path)
# print(f"Merged data with metadata saved to {output_path}")


In [ ]:
library(openxlsx)
library(dplyr)
library(ggplot2)
library(DESeq2)
library(lattice)
library(gplots)

expr_data_raw <- read.csv('H:/Projects/Integrative_Clustering/RNAseq/merged_expression_values_with_metadata_unstranded-counts.csv', row.names = 1, check.names = F)

expr_data <- expr_data_raw %>% filter(gene_type == "protein_coding")

expr_data <- expr_data %>% select(-gene_name, -gene_type)



# Create a "dummy" metadata table
metadata <- data.frame(row.names = colnames(expr_data))
# and dummy column because DESeq2 needs at least 1 column in the metadata
metadata$condition <- 1

# Create the DESeq2 dataset
dds <- DESeqDataSetFromMatrix(countData = expr_data, 
                              colData = metadata, 
                              design = ~ 1)


# Normalize and transform using rlog - This takes 5-10 minutes to run 
rlog_data <- rlog(dds, blind = TRUE)
rlog_expr_data <- assay(rlog_data)

# or simply load in the rlog_data to save some time
# write.csv(rlog_expr_data, "H:/Projects/Integrative_Clustering/RNAseq/rlog_expr_data.csv")
# rlog_expr_data <- read.csv("H:/Projects/Integrative_Clustering/RNAseq/rlog_expr_data.csv", row.names = 1, check.names = F)

# define function to calculate median absolute deviation
mad_function <- function(arr) {
  median(abs(arr - median(arr)))
}

# Optional variance filtering
# gene_mad <- apply(rlog_expr_data, 1, mad_function)
# mad_percentile <- 0.90
# mad_threshold <- quantile(gene_mad, mad_percentile)
# filt_rlog_expr_data <- rlog_expr_data[gene_mad >= mad_threshold, ]

# Flatten the dataset (convert to long format)
flattened_data <- as.vector(as.matrix(rlog_expr_data))

# Create a distribution plot
ggplot(data = data.frame(Value = flattened_data), aes(x = Value)) +
  geom_density(fill = "blue", alpha = 0.5) +
  labs(title = "Distribution of Flattened Dataset",
       x = "Values",
       y = "Density")


### Script Name: miRNA Processing

**Authors**: Joshua Bowers

**Language**: R

**Description**: This script loads already-preprocessed miRNA expression data and removes non-unique elements

**Requires**: Preprocess-miRNA expression matrix

In [ ]:
library(openxlsx)
library(dplyr)
library(ggplot2)
library(DESeq2)
library(lattice)
library(gplots)

mirna_data_raw <- read.xlsx('H:/Projects/Integrative_Clustering/miRNA/2negdCT.xlsx', sheet='Filtered log intensity')

# There seems to be some non-unique elemtns in the miRNA data. I want to remove those columns
# Identify non-unique elements
non_unique_ids <- unique(mirna_data_raw$UniqueID[duplicated(mirna_data_raw$UniqueID) | duplicated(mirna_data_raw$UniqueID, fromLast = TRUE)])

# Remove rows with non-unique elements
mirna_filtered <- mirna_data_raw[!mirna_data_raw$UniqueID %in% non_unique_ids, ]

# Make the UniqueIDs column the names of the rows
rownames(mirna_filtered) <- mirna_filtered$UniqueID

# select only numeric columns
mirna_data <- mirna_filtered %>% select(-UniqueID, -Missing, -`P-Value`, -Rank, -Variance, -`Num.1.5-Fold`, -Filter)


# There seems to be one missing value in miRNA_data_iclust at col 9 (0A$I42) row 381 (dme-mir-7-000268).
mirna_data <- mirna_data[rowSums(is.na(mirna_data)) == 0, ]

# Optional variance filtering
# mirna_gene_variances <- apply(mirna_data, 1, var) # calculate variance
# mirna_percentile <- 0.50 # set percentile - This keeps the 70th percentile and up (top 30% most variable miRNA)
# mirna_threshold <- quantile(mirna_gene_variances, mirna_percentile) # calculate numeric threshold representative of percentile
# mirna_data <- mirna_data[mirna_gene_variances >= mirna_threshold, ] # remove miRNA not exceeding (or equal to) the numeric threshold

# Flatten the dataset (convert to long format)
flattened_data <- as.vector(as.matrix(mirna_data))

# Create a distribution plot
ggplot(data = data.frame(Value = flattened_data), aes(x = Value)) +
  geom_density(fill = "blue", alpha = 0.5) +
  labs(title = "Distribution of Flattened Dataset",
       x = "Values",
       y = "Density")

### Script Name: Copy Number Pre-Processing

**Authors**: Joshua Bowers

**Language**: R

**Description**: This script loads in copy-number data and prepares it for downstream analysis. Specifically used to read in segement-level CN data from GISTIC2 segment/lesion-level CN matrix.

**Requires**: GISTIC2 segment/lesions output file

In [ ]:
library(openxlsx)
library(dplyr)
library(ggplot2)
library(DESeq2)
library(lattice)
library(gplots)

cn_type = 'seg'
if (cn_type == 'gene'){
  # Read in Copy Number Data (gene-level)
  cn_raw = read.csv("H:/Projects/Integrative_Clustering/Copy_Number/merged_copy_number_matrix.csv", row.names = 1, check.names = F)
  
  discrete <- T
  
  if (discrete == T){
    cn_discrete <- apply(cn_raw, c(1, 2), function(x) {
      if (x < 2) {
        return(0)  # Deletion
      } else if (x == 2) {
        return(1)   # Normal diploid
      } else if (x > 2 & x < 5) {
        return(2)   # Amplification
      } else if (x >= 5) {
        return(3)
      }
    })
    cn_data <- cn_discrete
    
    # Keep genes with CN alterations (non-diploid values) in ≥30% of samples
    alteration_freq <- apply(cn_data, 1, function(row) {
      mean(row != 1)  # Diploid is 2, so this counts amplifications and deletions
    })
    
  } else {
    cn_data <- cn_raw
    
    # Keep genes with CN alterations (non-diploid values) in ≥30% of samples
    alteration_freq <- apply(cn_data, 1, function(row) {
      mean(row != 2)  # Diploid is 1, so this counts amplifications and deletions
    })
    
  }
  
  altered_genes <- names(alteration_freq[alteration_freq >= 0.75])
  
  # Identify top 1000 most variable features
  cn_var <- apply(cn_data, 1, var)
  top_var_genes <- names(sort(cn_var, decreasing = TRUE)[1:1000])
  
  # Union of altered genes and top 1000 variable genes
  final_genes_to_keep <- union(altered_genes, top_var_genes)
  
  # Filter CN data to final selected genes
  cn_data_filt <- cn_data[final_genes_to_keep, ]
  
  # # calculate variance
  # cn_var = apply(cn_data, 1, var)
  # 
  # # set variance percentile to remove (this keeps only those values in the (x*100)th percentile - top (1.00-x)%)
  # cn_percentile <- 0.75
  # 
  # # calculate the numerical threshold of the desired percentile
  # cn_threshold <- quantile(cn_var, cn_percentile)
  # 
  # # filter out values that do not comply with desired threshold
  # cn_data_filt <- cn_data[cn_var >= cn_threshold, ]
} else if (cn_type == 'seg') {
  # read in alternative copy number data (segment-level CN)
  cn_raw = read.xlsx('H:/Projects/Integrative_Clustering/Copy_Number/GISTIC2_output_95/all_lesions_95_CN.xlsx', rowNames = T)
  cn_data_filt <- cn_raw[, -c(1:8)]
}

# Flatten the dataset (convert to long format)
flattened_data <- as.vector(as.matrix(cn_data_filt))

# Create a distribution plot
ggplot(data = data.frame(Value = flattened_data), aes(x = Value)) +
  geom_density(fill = "blue", alpha = 0.5) +
  labs(title = "Distribution of Flattened Dataset",
       x = "Values",
       y = "Density")

## Section 2: VMR Derivation and Exploration

### Script Name: VMR Derivation Analysis

**Authors**: Joshua Bowers

**Language**: R

**Description**: This script defines a list of variably methylated regions (VMRs) from a preprocessed methylation beta-value matrix using DMRcate's "variability" workflow.

**Requires**: TARGET sample phenotype matrix and preprocessed TARGET beta-value matrix

In [ ]:
# Load methylation values
meth_val_dir <- 'H:/Projects/Methylation_Project/TARGET_Reproducibility/reproduced_TARGET_funnorm_beta_filt.csv'

if(is.character(meth_val_dir) && length(meth_val_dir) == 1) {
    meth_vals <- fread(meth_val_dir)
    rownames(meth_vals) <- meth_vals$UniqueID
    meth_vals <- select(meth_vals, -UniqueID)
  } else if (is.data.frame(meth_val_dir) || is.matrix(meth_val_dir)) {
    meth_vals <- beta_val_dir
  } else {
    warning("You have entered an object that is not a data frame, matrix, or character for beta_val_dir!", immediate = T)
  }

# Load phenotype data
pheno_val_dir <- 'H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv'
phenotype_data <- fread(pheno_val_dir)
rownames(phenotype_data) <- phenotype_data$Sample_ID

# Create GRset
# set math_vals as matrix and set rownames of matrix to match the data.table
# This is needed for the cpg.annotate() method in DMRcate
meth_matrix <- as.matrix(meth_vals)
rownames(meth_matrix) <- rownames(meth_vals)

# Remove samples with no phenotype data
meth_matrix <- meth_matrix[, colnames(meth_matrix) %in% rownames(phenotype_data)]

# Create CpGannotated object for VMRs.  Necessary step for DMRcate pipeline
VMR_annotation <- cpg.annotate(datatype = "array", object = meth_matrix, what = "Beta", arraytype = "450K", analysis.type = "variability")

# Create a DMResults object for VMRs
VMRset <- dmrcate(VMR_annotation, lambda = 500, C = 2)

# Extract results from DMResults object
VMR_results <- extractRanges(VMRset, genome = "hg19")
print(paste("VMR:", length(VMR_results)))

# Optionally write to an output path
output_path <- '/output_example.csv'
write.csv(VMR_results, output_path)

### Script Name: Define VMR Band Information

*Authors*: Joshua Bowers

**Language**: R

**Description**: This script defines which VMRs belong to which chromosomal bands

**Requires**: UCSC reference genome chromosomal band information and global VMR profile

In [ ]:
##### Define Band Information for VMRs #####
library(dplyr)
library(GenomicRanges)

# load in band info found on UCSC genome browser
band_info <- read.csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/chr_band_info.csv')

# load in desired list of VMRs
VMRset <- read.csv("H:/Projects/Methylation_Project/TARGET_Continued/VMRs/vmr_using_dmrcate_beta_lam500.csv")

# Create genomic ranges for the band_info and VMRset data frames
gr_band_info <- GRanges(
  seqnames = Rle(band_info$chrom),
  ranges = IRanges(start = band_info$chromStart, end = band_info$chromEnd),
  name = paste(sub("chr", "", band_info$chrom), band_info$name, sep="")
)

gr_VMRset <- GRanges(
    seqnames = Rle(VMRset$chr),
    ranges = IRanges(start = VMRset$start, end = VMRset$end)
)

# Find the overlap between the two genomic ranges objects
overlaps <- findOverlaps(gr_VMRset, gr_band_info)

# Extract the concatenated band names that overlap with each VMR
VMRset$band <- rep(NA, length(gr_VMRset))  # Initialize the band column with NAs
hits <- queryHits(overlaps)
VMRset$band[hits] <- mapply(function(x) paste(gr_band_info$name[x], collapse=";"), subjectHits(overlaps), SIMPLIFY = FALSE)

# If multiple bands overlap a single VMR, the names will be concatenated together separated by semicolons
VMRset$band <- sapply(VMRset$band, function(x) if (is.null(x)) NA else paste(unique(x), collapse=";"))

# Optionally, write band info to CSV
# write.csv(VMRset, 'H:/Projects/Methylation_Project/TARGET_Continued/VMRs/vmr_using_dmrcate_beta_lam500_band_info.csv', row.names = F)

### Script Name: VMR Distribution Exploration

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script explores chromosome-level distribution and enrichment of the VMR global profile

**Requires**: VMR global profile (M-values) and an annotation file containing all CpG's found in the Illumina 450k array

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import numpy as np
from scipy.stats import chi2

# Read in vmr information
vmr_df = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv', index_col=0)

# Parse out the chromosome number from "chr" column
vmr_df['chr_num'] = vmr_df['chr'].str.extract(r'(\d+)').astype(int)

# Sort by chromosome and start position
vmr_df = vmr_df.sort_values(['chr_num', 'start'])

# Read in all CpG probes that passed QC - This is only needed if doing per CpG normalization
cpg_df = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/anno.csv', index_col=3)

# Pulls chr number as int
cpg_df['chr_num'] = (
    cpg_df['annotation_data.chr']
      .astype(str)
      .str.extract(r'(\d+)')[0]
      .astype('Int64')
)

# Define chromosome sizes (in base pairs)
chr_sizes = {
    1: 248956422, 2: 242193529, 3: 198295559, 4: 190214555,
    5: 181538259, 6: 170805979, 7: 159345973, 8: 145138636,
    9: 138394717, 10: 133797422, 11: 135086622, 12: 133275309,
    13: 114364328, 14: 107043718, 15: 101991189, 16: 90338345,
    17: 83257441, 18: 80373285, 19: 58617616, 20: 64444167,
    21: 46709983, 22: 50818468
}

# Count VMRs per chromosome
vmr_counts = vmr_df['chr_num'].value_counts().sort_index()

# Convert to DataFrame and normalize by chromosome size (per Mb)
density_df = pd.DataFrame({
    'chr_num': vmr_counts.index,
    'vmr_count': vmr_counts.values,
    'chr_size_bp': [chr_sizes[c] for c in vmr_counts.index]
})
density_df['vmrs_per_mb'] = density_df['vmr_count'] / (density_df['chr_size_bp'] / 1e6)

# Set style and context
plt.figure(figsize=(12, 6))  # Wider for readability

# Barplot
ax = sns.barplot(
    data=density_df,
    x='chr_num',
    y='vmrs_per_mb',
    edgecolor='black'
)

# Axis titles
ax.set_title("VMR Density per Chromosome", fontsize=18, weight='bold', pad=15)
ax.set_xlabel("Chromosome", fontsize=14)
ax.set_ylabel("VMRs per Megabase", fontsize=14)

# Tick customization
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Remove top/right spines
sns.despine()

# Add value labels above bars
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3, fontsize=11)

# Tight layout
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import poisson
from statsmodels.stats.multitest import multipletests
import numpy as np


covered_cpgs_per_chr = (
    cpg_df['chr_num']
      .value_counts()
      .rename_axis('chr_num')
      .sort_index()
      .astype(int)
)

##### prep per-chromosome counts #####
# list of all chr numbers
autosomes = pd.Index(range(1, 23), name='chr_num')

# Finds VMR counts per chr
vmr_counts = (vmr_df.loc[vmr_df['chr_num'].between(1, 22), 'chr_num']
              .value_counts().reindex(autosomes, fill_value=0).astype(int))

# Translates chr to a number
cpg_chr = (cpg_df['annotation_data.chr'].astype(str)
           .str.extract(r'(\d+)')[0].astype('Int64'))

# finds CpGs per chr
covered_cpgs = (cpg_chr[cpg_chr.between(1, 22)]
                .value_counts().reindex(autosomes, fill_value=0).astype(int))

# ##### expected counts and O/E #####
N = int(vmr_counts.sum()) # overall number of VMRs
total_cov = int(covered_cpgs.sum()) # total number of CpGs
p = covered_cpgs / total_cov # per chr ratio of VMRs sites to total CpGs
expected = N * p # expected number of VMRs per CpG site

obs = vmr_counts.values.astype(int) # actual (observed) number of VMRs sites 
exp = expected.values # expected number of VMRs per CpG site

OE = obs / exp # ratio of observed to expected VMRs

# ##### Garwood 95% Poisson CIs on counts, scaled to O/E #####
alpha = 0.05
lower_counts = 0.5 * chi2.ppf(alpha/2, 2*obs)
upper_counts = 0.5 * chi2.ppf(1 - alpha/2, 2*(obs + 1))

OE_low = np.divide(lower_counts, exp, out=np.full_like(exp, np.nan, dtype=float), where=exp>0)
OE_high = np.divide(upper_counts, exp, out=np.full_like(exp, np.nan, dtype=float), where=exp>0)

results = pd.DataFrame({
    'chr_num': autosomes,
    'covered_cpgs': covered_cpgs.values,
    'vmr_count': vmr_counts.values,
    'expected_vmrs': expected.values,
    'OE': OE,
    'OE_low95': OE_low,
    'OE_high95': OE_high
})

def poisson_twosided_p(k, lam):
    # tail probs
    lower = poisson.cdf(k, lam)
    upper = 1 - poisson.cdf(k-1, lam)  # = sf(k-1, lam)
    p = 2 * min(lower, upper)
    return min(p, 1.0)

results['p_poisson'] = [
    poisson_twosided_p(int(k), float(lam))
    for k, lam in zip(results['vmr_count'], results['expected_vmrs'])
]
results['p_poisson_fdr'] = multipletests(results['p_poisson'], method='fdr_bh')[1]

# Was interacting strangly with previous sections
mpl.rcParams.update(mpl.rcParamsDefault)   # hard reset rcParams
plt.style.use("default")                   # reset style

mask = np.isfinite(results['OE'])
x = results.loc[mask, 'chr_num'].values
y = results.loc[mask, 'OE'].values
yerr = np.vstack([
    y - results.loc[mask, 'OE_low95'].values,
    results.loc[mask, 'OE_high95'].values - y
])

plt.figure(figsize=(13,4))
plt.errorbar(x, y, yerr = yerr, fmt = 'o', capsize = 3, color = 'teal')
plt.axhline(1.0, ls='--', lw=1, color = 'black')
plt.xlim(0.5, 22.5)
plt.xticks(autosomes, autosomes, fontsize=14, fontweight='bold')
plt.yticks(fontsize=14)
plt.xlabel("Chromosome")
plt.ylabel("Observed / Expected (by CpG coverage)")
plt.title("Chromosome-level VMR enrichment with 95% CIs")
plt.tight_layout()
plt.show()

In [ ]:
##### VMR Count by Chr #####
ordered_chrs = list(chr_sizes.keys())

# === Filter and sort ===
vmr_df = vmr_df[vmr_df['chr_num'].isin(ordered_chrs)].copy()
vmr_df['chr_num'] = pd.Categorical(vmr_df['chr_num'], categories=ordered_chrs, ordered=True)
vmr_df = vmr_df.sort_values(['chr_num', 'start'])

##### Compute genome-wide position #####
chr_offsets = {}
offset = 0
for chr_ in ordered_chrs:
    chr_offsets[chr_] = offset
    offset += chr_sizes[chr_]

vmr_df['genome_start'] = vmr_df.apply(lambda row: row['start'] + chr_offsets[row['chr_num']], axis=1)

# === Plot bar chart ===
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(14.5, 4))  # wider height

# Set fixed bin edges to avoid extra whitespace on each side of the figure
genome_length = sum(chr_sizes.values())
num_bins = 200
bin_edges = pd.IntervalIndex.from_breaks(
    np.linspace(0, genome_length, num_bins + 1), closed='left'
)

# Plot histogram
sns.histplot(
    vmr_df['genome_start'],
    bins=bin_edges.left.values,
    color='teal',
    edgecolor=None,
    ax=ax
)

# Title and labels
#ax.set_title("Global Distribution of VMRs", fontsize=14, fontweight='bold')
ax.set_xlabel("Genomic Position", fontsize=16, fontweight='bold')
ax.set_ylabel("VMR Count", fontsize=16, fontweight='bold')

# Set x-ticks at chromosome starts
chr_ticks = []
chr_labels = []
offset = 0
for chr_ in ordered_chrs:
    chr_ticks.append(offset)
    chr_labels.append(chr_)
    offset += chr_sizes[chr_]

ax.set_xticks(chr_ticks)
ax.set_xticklabels(chr_labels, rotation=90)

# Optional: vertical lines at chromosome boundaries
offset = 0
for chr_ in ordered_chrs:
    offset += chr_sizes[chr_]
    ax.axvline(offset, color='lightgray', linestyle='--', linewidth=0.5)

# Clean up spacing
ax.set_xlim(0, genome_length)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Enhancer Analysis
# Load inputs, identify enhancer-associated VMRs, and perform
# UMAP/DBSCAN clustering with downstream RFS analyses.
# ============================================================

library(ggplot2)
library(dplyr)
library(tidyverse)
library(readxl)
library(ggpubr)
library(survminer)
library(survival)
library(broom)
library(FSA)
library(RColorBrewer)
library(viridisLite)
library(uwot)
library(FactoMineR)
library(dbscan)
library(GenomicRanges)

# ============================================================
# Input loading
# Read VMR subgroup definitions, global VMR methylation data,
# TARGET metadata, and enhancer consensus peak annotations.
# ============================================================

vmr_grp <- read_xlsx("../../Input/resort_type_vmr_subgroupings.xlsx")
vmr_data <- read.csv("../../Input/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv")
target_meta <- read.csv("../../Input/TARGET_meta_data.csv")
enhancer_data <- read_xlsx("../Input/collapsed_majoity_consensus_peaks.xlsx")
enhancer_data <- as.data.frame(enhancer_data)

# ============================================================
# Enhancer-associated VMR mapping
# Identify VMRs that directly overlap enhancer regions and
# VMRs that fall within 1 kb of enhancer loci.
# ============================================================

enhancer_data$chr <- enhancer_data$seqnames
enhancer_data$number <- rownames(enhancer_data)

# Convert enhancer and VMR coordinates to GRanges objects
enhancers <- with(enhancer_data, GRanges(seqnames = chr, ranges = IRanges(start = start, end = end)))
vmrs <- with(vmr_data, GRanges(seqnames = chr, ranges = IRanges(start = start, end = end)))

# VMRs that directly overlap at least one enhancer region
overlap_hits <- findOverlaps(vmrs, enhancers)
vmrs_overlapping <- vmr_data[queryHits(overlap_hits), ]

# VMRs located within 1 kb of enhancer regions, including direct overlaps
enhancers_expanded <- resize(enhancers, width = width(enhancers) + 2000, fix = "center")
proximity_hits <- findOverlaps(vmrs, enhancers_expanded)
vmrs_within_1000bp <- vmr_data[queryHits(proximity_hits), ]

# ============================================================
# Enhancer-associated VMR subset construction
# Retain VMR IDs and sample methylation columns, then generate
# overlap- and proximity-based VMR subsets split by CpG context.
# ============================================================

# Retain VMR identifiers and sample-level methylation columns
vmr_filt <- vmr_data %>% select(1, 12:94)

# Subset to enhancer-associated VMRs using unique VMR IDs
# This also removes duplicate overlap calls from VMRs matched
# to multiple enhancer regions
vmr_overlap <- vmr_filt %>% filter(VMR_ID %in% vmrs_overlapping$VMR_ID)
vmr_1k <- vmr_filt %>% filter(VMR_ID %in% vmrs_within_1000bp$VMR_ID)

# Split enhancer-associated VMR sets by CpG-context subgroup
vmr_island_overlap <- vmr_overlap %>% filter(VMR_ID %in% vmr_grp$filtered_Group1island.VMR_ID)
vmr_shore_overlap <- vmr_overlap %>% filter(VMR_ID %in% vmr_grp$filtered_Group2shore.VMR_ID)
vmr_opensea_overlap <- vmr_overlap %>% filter(VMR_ID %in% vmr_grp$filtered_Group3opensea.VMR_ID)
vmr_shelf_overlap <- vmr_overlap %>% filter(VMR_ID %in% vmr_grp$filtered_Group4shelf.VMR_ID)

vmr_island_1k <- vmr_1k %>% filter(VMR_ID %in% vmr_grp$filtered_Group1island.VMR_ID)
vmr_shore_1k <- vmr_1k %>% filter(VMR_ID %in% vmr_grp$filtered_Group2shore.VMR_ID)
vmr_opensea_1k <- vmr_1k %>% filter(VMR_ID %in% vmr_grp$filtered_Group3opensea.VMR_ID)
vmr_shelf_1k <- vmr_1k %>% filter(VMR_ID %in% vmr_grp$filtered_Group4shelf.VMR_ID)

# ============================================================
# All enhancer-overlapping VMRs
# UMAP projection, DBSCAN clustering, and RFS analysis.
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_overlap[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.001,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/overlap_8_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

# DBSCAN clustering on UMAP coordinates
umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.7, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/vmr_overlap_8_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/umap_overlap_8_001_df.csv")

# Survival analysis
umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Overlap VMR Cluster")

pdf("../Output/surv_overlap.pdf")
RFS_plot
dev.off()

# Stratify by metastasis at diagnosis
umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Overlap VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/surv_overlap_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-overlapping island VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_island_overlap[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.001,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/Island/island_overlap_8_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.72, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/VMR type subsets/Island/vmr_island_overlap_8_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/Island/umap_island_overlap_8_001_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island Overlap VMR Cluster")

pdf("../Output/VMR type subsets/Island/surv_island_overlap.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island Overlap VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/Island/surv_island_overlap_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-overlapping shore VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_shore_overlap[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.01,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/Shore/shore_overlap_8_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.7, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/VMR type subsets/Shore/vmr_shore_overlap_8_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/Shore/umap_shore_overlap_8_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore Overlap VMR Cluster")

pdf("../Output/VMR type subsets/Shore/surv_shore_overlap.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore Overlap VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/Shore/surv_shore_overlap_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-overlapping open sea VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_opensea_overlap[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.01,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/Opensea/opensea_overlap_8_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.7, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/VMR type subsets/Opensea/vmr_opensea_overlap_8_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/opensea/umap_opensea_overlap_8_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("opensea Overlap VMR Cluster")

pdf("../Output/VMR type subsets/opensea/surv_opensea_overlap.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("opensea Overlap VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/opensea/surv_opensea_overlap_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-overlapping shelf VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_shelf_overlap[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 15, min_dist = 0.01,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/shelf/shelf_overlap_15_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 3, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/VMR type subsets/shelf/vmr_shelf_overlap_15_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/shelf/umap_shelf_overlap_15_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("shelf Overlap VMR Cluster")

pdf("../Output/VMR type subsets/shelf/surv_shelf_overlap.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("shelf Overlap VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/shelf/surv_shelf_overlap_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# All enhancer-proximal (within 1 kb) VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_1k[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.001,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/1k_overlap_8_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.67, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)
umap_df$Cluster_1_3_merge <- ifelse(umap_df$Cluster == 3, 1, umap_df$Cluster)

pdf("../Output/1k_vmr_overlap_8_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster_1_3_merge)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/1k_umap_overlap_8_001_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#79C97E", "#9EBBF8"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Overlap VMR Cluster")

rfs_13.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster_1_3_merge, umap_df_surv)
RFS_13_plot <- ggsurvplot(
  rfs_13.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Overlap VMR Cluster")

pdf("../Output/1k_surv_overlap.pdf")
RFS_plot
RFS_13_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster_1_3_merge == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster_1_3_merge == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster_1_3_merge == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster_1_3_merge == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Overlap VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/1k_surv_overlap_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-proximal island VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_island_1k[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.001,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/Island/island_1k_8_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.525, minPts = 8)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/VMR type subsets/Island/vmr_island_1k_8_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/Island/umap_island_1k_8_001_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#79C97E", "#9EBBF8"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island 1k VMR Cluster")

pdf("../Output/VMR type subsets/Island/surv_island_1k.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    Cluster == 0 & Met_at_Dx == 1 ~ "Cluster 0, Met Pos",
    Cluster == 0 & Met_at_Dx == 0 ~ "Cluster 0, Met Neg",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#79C97E", "lightgreen", "#9EBBF8", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island 1k VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/Island/surv_island_1k_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-proximal shore VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_shore_1k[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.001,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/Shore/shore_1k_8_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.7, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/VMR type subsets/Shore/vmr_shore_1k_8_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/Shore/umap_shore_1k_8_001_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore 1k VMR Cluster")

pdf("../Output/VMR type subsets/Shore/surv_shore_1k.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore 1k VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/Shore/surv_shore_1k_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-proximal open sea VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_opensea_1k[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.01,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/Opensea/opensea_1k_8_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.6, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)
umap_df$Cluster_0_2_merge <- ifelse(umap_df$Cluster == 0, 2, umap_df$Cluster)

pdf("../Output/VMR type subsets/Opensea/vmr_opensea_1k_8_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster_0_2_merge)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/opensea/umap_opensea_1k_8_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster_0_2_merge, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("opensea 1k VMR Cluster")

pdf("../Output/VMR type subsets/opensea/surv_opensea_1k.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster_0_2_merge == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster_0_2_merge == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster_0_2_merge == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster_0_2_merge == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("opensea 1k VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster_0_2_merge + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/opensea/surv_opensea_1k_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Enhancer-proximal shelf VMRs
# The original file clipped at the end of this block, so this
# final subsection follows the same structure used above.
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_shelf_1k[, -1]))

set.seed(42)
umap_result <- umap(vmr_matrix, n_neighbors = 8, min_dist = 0.001,
                    metric = "euclidean", n_components = 2, scale = FALSE)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/VMR type subsets/shelf/shelf_1k_8_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 1, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/VMR type subsets/shelf/vmr_shelf_1k_8_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data",
       x = "UMAP 1",
       y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/VMR type subsets/shelf/umap_shelf_1k_8_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shelf 1k VMR Cluster")

pdf("../Output/VMR type subsets/shelf/surv_shelf_1k.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shelf 1k VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/VMR type subsets/shelf/surv_shelf_1k_strat.pdf")
RFS_strat_plot
dev.off()

### Script Name: VMR resort type Subgrouping UMAP and Survival Analysis

**Authors**: Miya Hugaboom

**Language**: R

**Description**: This script summarizes the distribution of VMRs across CpG-context subgroups, subsets the global TARGET VMR methylation matrix into those subgroup-specific profiles, and performs subgroup-level unsupervised clustering using UMAP and DBSCAN. For each subgroup, the resulting clusters are then evaluated for association with recurrence-free survival, including analyses stratified by metastasis at diagnosis

**Requires**: TARGET global VMR methylation matrix, VMR subgroup annotation table by CpG-context, TARGET sample phenotype metadata

In [ ]:
# ============================================================
# Sarcoma VMR Subgroup Analysis
# Load subgroup assignments, summarize VMR subgroup counts,
# and perform subgroup-specific UMAP/DBSCAN clustering with
# downstream recurrence-free survival analyses.
# ============================================================

library(ggplot2)
library(dplyr)
library(tidyverse)
library(readxl)
library(ggpubr)
library(survminer)
library(survival)
library(broom)
library(FSA)
library(RColorBrewer)
library(viridisLite)
library(uwot)
library(FactoMineR)
library(dbscan)

# ============================================================
# Input loading
# Read subgroup annotations, VMR methylation matrix, and
# TARGET sample metadata.
# ============================================================

vmr_grp <- read_xlsx("../Input/resort_type_vmr_subgroupings.xlsx")
vmr_data <- read.csv("../Input/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv")
target_meta <- read.csv("../Input/TARGET_meta_data.csv")

# ============================================================
# VMR subgroup distribution
# Count the number of VMRs assigned to each subgroup and
# generate a summary bar plot.
# ============================================================

vmr_counts <- sapply(vmr_grp, function(col) sum(!is.na(col)))

vmr_summary <- data.frame(
  Subgroup = names(vmr_counts),
  Count = as.integer(vmr_counts)
)

vmr_summary$Name <- c(
  "Island", "Shore", "OpenSea", "Shelf",
  "Island/Shore", "Shore/Shelf", "OpenSea/Shelf", "Multiple (>2)"
)

custom_colors <- c(
  "Island" = "#FB8072",
  "Shore" = "#7BAFDE",
  "OpenSea" = "#FAE174",
  "Shelf" = "#E78AC3",
  "Island/Shore" = "#ba5ce3",
  "Shore/Shelf" = "#E7298A",
  "OpenSea/Shelf" = "#64C870",
  "Multiple (>2)" = "#641400"
)

vmr_summary$Name <- factor(
  vmr_summary$Name,
  levels = c("Island", "Shore", "OpenSea", "Shelf",
             "Island/Shore", "Shore/Shelf", "OpenSea/Shelf", "Multiple (>2)")
)

pdf("../Sarcoma/Output/vmr_group_bar_20241111.pdf")
ggplot(vmr_summary, aes(x = Name, y = Count, fill = Name)) +
  geom_bar(stat = "identity") +
  theme_minimal() +
  labs(
    title = "Distribution of VMRs by Subgroup",
    x = "VMR Subgroup",
    y = "Number of VMRs"
  ) +
  theme(axis.text.x = element_text(angle = 0, hjust = 1, vjust = 0.5)) +
  scale_fill_manual(values = custom_colors) +
  theme_classic() +
  scale_y_continuous(expand = c(0, 0))
dev.off()

# ============================================================
# VMR subgroup subset construction
# Retain VMR IDs and sample methylation columns, then split
# the global VMR matrix into CpG-context subgroups.
# ============================================================

vmr_filt <- vmr_data %>% select(1, 12:94)

vmr_island  <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group1island.VMR_ID)
vmr_shore   <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group2shore.VMR_ID)
vmr_opensea <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group3opensea.VMR_ID)
vmr_shelf   <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group4shelf.VMR_ID)
vmr_ixs     <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group5islexandshore.VMR_ID)
vmr_sxs     <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group6shoreshelf.VMR_ID)
vmr_oxs     <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group7openseashelf.VMR_ID)
vmr_multi   <- vmr_filt %>% filter(VMR_ID %in% vmr_grp$filtered_Group8other.VMR_ID)

# ============================================================
# Island VMRs
# UMAP projection, DBSCAN clustering, and RFS analysis.
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_island[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 15,
  min_dist = 0.001,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Island/vmr_island_15_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.6, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/Island/vmr_island_15_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/Island/umap_15_001_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island VMR Cluster")

pdf("../Output/Island/surv_island.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/Island/surv_island_strat.pdf")
RFS_strat_plot
dev.off()

# Alternate island UMAP settings explored but not selected
set.seed(42)
umap_result_scaled <- umap(
  vmr_matrix,
  n_neighbors = 8,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = TRUE
)

umap_df_scaled <- data.frame(
  UMAP1 = umap_result_scaled[, 1],
  UMAP2 = umap_result_scaled[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Island/vmr_island_scale_8_01.pdf")
ggplot(umap_df_scaled, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

set.seed(42)
umap_result_pca <- umap(
  vmr_matrix,
  n_neighbors = 8,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE,
  pca = 50
)

umap_df_pca <- data.frame(
  UMAP1 = umap_result_pca[, 1],
  UMAP2 = umap_result_pca[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Island/vmr_island_pca_8_01.pdf")
ggplot(umap_df_pca, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

set.seed(42)
umap_result_pca <- umap(
  vmr_matrix,
  n_neighbors = 15,
  min_dist = 0.001,
  metric = "euclidean",
  n_components = 2,
  scale = TRUE,
  pca = 50
)

umap_df_pca <- data.frame(
  UMAP1 = umap_result_pca[, 1],
  UMAP2 = umap_result_pca[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Island/vmr_island_pca_scale_15_001.pdf")
ggplot(umap_df_pca, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

# ============================================================
# Shore VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_shore[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 15,
  min_dist = 0.001,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Shore/vmr_shore_15_001.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.6, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/Shore/vmr_shore_15_001_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/Shore/umap_15_001_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore VMR Cluster")

pdf("../Output/Shore/surv_shore.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/Shore/surv_shore_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Open sea VMRs
# Two candidate UMAP parameterizations retained.
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_opensea[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 15,
  min_dist = 0.001,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df_15_001 <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Opensea//vmr_opensea_15_001.pdf")
ggplot(umap_df_15_001, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 10,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df_10_01 <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Opensea//vmr_opensea_10_01.pdf")
ggplot(umap_df_10_01, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords_15_001 <- umap_df_15_001[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords_15_001, eps = 0.55, minPts = 5)
umap_df_15_001$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/Opensea/vmr_opensea_15_001_dbscan.pdf")
ggplot(umap_df_15_001, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df_15_001, "../Output/Opensea/umap_15_001_df.csv")

umap_coords_10_01 <- umap_df_10_01[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords_10_01, eps = 0.68, minPts = 5)
umap_df_10_01$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/Opensea/vmr_opensea_10_01_dbscan.pdf")
ggplot(umap_df_10_01, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df_10_01, "../Output/Opensea/umap_10_01_df.csv")

umap_df_15_001$Sample_ID <- gsub("X0", "0", umap_df_15_001$Sample)
umap_df_15_001_surv <- left_join(umap_df_15_001, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_15_001_surv)
RFS_plot_15_001 <- ggsurvplot(
  rfs.uni,
  data = umap_df_15_001_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("OpenSea VMR Cluster")

pdf("../Output/Opensea/surv_opensea_15_001.pdf")
RFS_plot_15_001
dev.off()

umap_df_15_001_surv <- umap_df_15_001_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs_15_001.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_15_001_surv)
RFS_strat_plot <- ggsurvplot(
  rfs_15_001.strat.uni,
  data = umap_df_15_001_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("OpenSea VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_15_001_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_15_001_surv)

pdf("../Output/Opensea/surv_opensea_strat_15_001.pdf")
RFS_strat_plot
dev.off()

umap_df_10_01$Sample_ID <- gsub("X0", "0", umap_df_10_01$Sample)
umap_df_10_01_surv <- left_join(umap_df_10_01, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_10_01_surv)
RFS_plot_10_01 <- ggsurvplot(
  rfs.uni,
  data = umap_df_10_01_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("OpenSea VMR Cluster")

pdf("../Output/Opensea/surv_opensea_10_01.pdf")
RFS_plot_10_01
dev.off()

umap_df_10_01_surv <- umap_df_10_01_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs_10_01.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_10_01_surv)
RFS_strat_plot <- ggsurvplot(
  rfs_10_01.strat.uni,
  data = umap_df_10_01_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("OpenSea VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_10_01_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_10_01_surv)

pdf("../Output/Opensea/surv_opensea_strat_10_01.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Shelf VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_shelf[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 15,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Shelf/vmr_shelf_15_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.51, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)
umap_df$Cluster_0_1_merge <- ifelse(umap_df$Cluster == 0, 1, umap_df$Cluster)
umap_df$Cluster_0_1_3_merge <- ifelse(umap_df$Cluster == 0 | umap_df$Cluster == 3, 1, umap_df$Cluster)

pdf("../Output/Shelf/vmr_shelf_15_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster_0_1_merge)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster_0_1_3_merge)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/Shelf/umap_15_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs_01.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster_0_1_merge, umap_df_surv)
RFS_01_plot <- ggsurvplot(
  rfs_01.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#79C97E", "#9EBBF8"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shelf VMR Cluster")

rfs_013.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster_0_1_3_merge, umap_df_surv)
RFS_013_plot <- ggsurvplot(
  rfs_013.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shelf VMR Cluster")

pdf("../Output/Shelf/surv_shelf.pdf")
RFS_01_plot
RFS_013_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster_0_1_3_merge == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster_0_1_3_merge == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster_0_1_3_merge == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster_0_1_3_merge == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shelf VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster_0_1_3_merge + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/Shelf/surv_shelf_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Island/Shore VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_ixs[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 10,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/IslandShore/vmr_ixs_10_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.5, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/IslandShore/vmr_ixs_10_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/IslandShore/umap_10_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")
umap_df_surv$Cluster_merge <- ifelse(
  umap_df_surv$Cluster == 0 | umap_df_surv$Cluster == 3 | umap_df_surv$Cluster == 4,
  "Other",
  umap_df_surv$Cluster
)

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#BDBE60", "#7CCEA7", "#76C4F4", "#E09FF1"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island-Shore VMR Cluster")

rfs_merge.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster_merge, umap_df_surv)
RFS_merge_plot <- ggsurvplot(
  rfs_merge.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#BDBE60", "#7CCEA7", "grey"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island-Shore VMR Cluster")

pdf("../Output/IslandShore/surv_ixs.pdf")
RFS_plot
RFS_merge_plot
dev.off()

umap_df_surv$Met_Strat_Group <- interaction(umap_df_surv$Cluster, umap_df_surv$Met_at_Dx)
umap_df_surv$Met_Strat_Group <- factor(
  umap_df_surv$Met_Strat_Group,
  levels = c("0.0", "0.1", "1.0", "1.1", "2.0", "2.1", "3.0", "3.1", "4.0", "4.1")
)

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#BDBE60", "yellowgreen", "#7CCEA7", "lightgreen", "#76C4F4", "lightblue1", "#E09FF1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Island-Shore VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/IslandShore/surv_ixs_strat.pdf", height = 10, width = 10)
RFS_strat_plot
dev.off()

# ============================================================
# Shore/Shelf VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_sxs[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 15,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/ShoreShelf/vmr_shoreshelf_15_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.58, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/ShoreShelf/vmr_sxs_15_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/ShoreShelf/umap_15_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore-Shelf VMR Cluster")

pdf("../Output/ShoreShelf/surv_sxs.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Shore-Shelf VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/ShoreShelf//surv_sxs_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# OpenSea/Shelf VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_oxs[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 8,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/OpenseaShelf/vmr_openseashelf_8_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 0.6, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/OpenseaShelf/vmr_oxs_8_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/OpenseaShelf/umap_8_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Opensea-Shelf VMR Cluster")

pdf("../Output/OpenseaShelf/surv_oxs.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("OpenSea-Shelf VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/OpenseaShelf//surv_oxs_strat.pdf")
RFS_strat_plot
dev.off()

# ============================================================
# Multiple-context VMRs
# ============================================================

vmr_matrix <- as.data.frame(t(vmr_multi[, -1]))

set.seed(42)
umap_result <- umap(
  vmr_matrix,
  n_neighbors = 8,
  min_dist = 0.01,
  metric = "euclidean",
  n_components = 2,
  scale = FALSE
)

umap_df <- data.frame(
  UMAP1 = umap_result[, 1],
  UMAP2 = umap_result[, 2],
  Sample = rownames(vmr_matrix)
)

pdf("../Output/Multi//vmr_multi_8_01.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

umap_coords <- umap_df[, c("UMAP1", "UMAP2")]
set.seed(42)
dbscan_result <- dbscan(umap_coords, eps = 1, minPts = 5)

umap_df$Cluster <- as.character(dbscan_result$cluster)

pdf("../Output/Multi/vmr_multi_8_01_dbscan.pdf")
ggplot(umap_df, aes(x = UMAP1, y = UMAP2, label = Sample, color = Cluster)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_text(vjust = 1, hjust = 1, size = 2) +
  theme_minimal() +
  labs(title = "UMAP Projection of VMR Data", x = "UMAP 1", y = "UMAP 2")
dev.off()

write.csv(umap_df, "../Output/Multi//umap_8_01_df.csv")

umap_df$Sample_ID <- gsub("X0", "0", umap_df$Sample)
umap_df_surv <- left_join(umap_df, target_meta, by = "Sample_ID")

rfs.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Cluster, umap_df_surv)
RFS_plot <- ggsurvplot(
  rfs.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "#7BCFD4"),
  pval = TRUE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Multi VMR Cluster")

pdf("../Output/Multi/surv_multi.pdf")
RFS_plot
dev.off()

umap_df_surv <- umap_df_surv %>%
  mutate(Met_Strat_Group = case_when(
    Cluster == 1 & Met_at_Dx == 0 ~ "Cluster 1, Met Neg",
    Cluster == 2 & Met_at_Dx == 0 ~ "Cluster 2, Met Neg",
    Cluster == 1 & Met_at_Dx == 1 ~ "Cluster 1, Met Pos",
    Cluster == 2 & Met_at_Dx == 1 ~ "Cluster 2, Met Pos",
    TRUE ~ NA_character_
  ))

rfs.strat.uni <- survfit(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, umap_df_surv)
RFS_strat_plot <- ggsurvplot(
  rfs.strat.uni,
  data = umap_df_surv,
  ylab = "Recurrence-free survival",
  palette = c("#E4897D", "pink", "#7BCFD4", "lightblue1"),
  pval = FALSE,
  risk.table = TRUE,
  legend = "none"
) + ggtitle("Multi VMR Cluster, MetDx")

logrank_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Met_Strat_Group, data = umap_df_surv)
logrank_strat_results <- survdiff(Surv(RFS_months, RFS_cens) ~ Cluster + strata(Met_at_Dx), data = umap_df_surv)

pdf("../Output/Multi/surv_mutli_strat.pdf")
RFS_strat_plot
dev.off()

### Script Name: Enhancer-Associated VMR Mapping

**Authors**: Miya Hugaboom

**Language**: R

**Description**: This script identifies VMRs from the TARGET global methylation profile that directly overlap enhancer regions or lie within 1 kb of enhancers, then subsets these enhancer-associated VMRs by CpG-context category for downstream clustering and survival analyses.

**Requires**: TARGET global VMR methylation matrix, VMR subgroup annotation table, TARGET sample phenotype metadata, enhancer consensus peak annotations

## Section 3: Unsupervised Analysis

### Script Name: Unsupervised UMAP Clustering

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script runs an unsupervised analysis of global VMR profile by utilizing UMAP coupled with k-means clustering to define two molecularly distinct global clusters.

**Requires**: TARGET sample phenotype matrix and global VMR profile (M-values)

In [ ]:
from umap import UMAP
from sklearn.cluster import KMeans
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# loading in phenotypic data
pheno = pd.read_csv('/Volumes/homedir$/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv')
pheno = pheno.set_index('Sample_ID')

# load in methylation data
vmr_df = pd.read_excel('/Volumes/homedir$/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/sampleAvg_ONLY_vmr_using_dmrcate_beta_lam500.xlsx', index_col=0).T
vmr_df.sort_index(inplace=True)

# Run UMAP
umap_model = UMAP(n_neighbors=8, min_dist=0.01, n_components=2, random_state=1)
umap_results = umap_model.fit_transform(vmr_df)

# Create DataFrame
umap_df = pd.DataFrame(umap_results, index=vmr_df.index, columns=['UMAP_1', 'UMAP_2'])

# KMeans Clustering
kmeans = KMeans(n_clusters=2, random_state=42)
umap_df['KMeans_cluster'] = kmeans.fit_predict(umap_df[['UMAP_1', 'UMAP_2']]) + 1 # add 1 to match previous clustering characterization

# Define a color palette
palette = ['#1f77b4', '#ff7f0e']

plt.figure(figsize=(10, 8))

# Plot each cluster individually with its own color
for cluster in sorted(umap_df['KMeans_cluster'].unique()):
    cluster_points = umap_df[umap_df['KMeans_cluster'] == cluster][['UMAP_1', 'UMAP_2']]
    
    # Plot the points
    plt.scatter(
        cluster_points['UMAP_1'],
        cluster_points['UMAP_2'],
        color=palette[cluster - 1],
        label=f'Cluster {cluster}',
        s=50,  # size of points
        alpha=0.7
    )
    
    # Draw convex hull
    if len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        hull_pts = np.append(hull.vertices, hull.vertices[0])  # loop back
        plt.plot(
            cluster_points.values[hull_pts, 0],
            cluster_points.values[hull_pts, 1],
            linestyle='--',
            linewidth=2,
            color=palette[cluster - 1]
        )

# Plot styling
plt.title(' ')
plt.xlabel('UMAP Dimension 1', fontsize=16, fontweight='bold')
plt.ylabel('UMAP Dimension 2', fontsize=16, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')  
plt.yticks(fontsize=14, fontweight='bold')
plt.legend(fontsize=16)
plt.tight_layout()
plt.show()

### Script Name: Generate UMAP Cluster Heatmap

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script generates a heatmap containing extensive phenotyic information and sorts samples by average methylation (left = high, right = low)

**Requires**: TARGET sample phenotype matrix and global VMR profile (M-values)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.stats import zscore
import matplotlib.colors as mcolors

# Load the files
pheno = pd.read_csv('/Volumes/homedir$/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)
vmr_data = pd.read_csv('/Volumes/homedir$/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv', index_col=0)

# Extract relevant columns from TARGET_Methylation.csv
methylation_clusters = pheno[['eyeball_umap_clusters']]

# Ensure that the chromosome sorting is correct by handling chromosomes as strings
vmr_data['chr_num'] = vmr_data['chr'].str.extract(r'(\d+)', expand=False).fillna(vmr_data['chr'])
vmr_data['chr_num'] = vmr_data['chr_num'].apply(lambda x: int(x) if x.isdigit() else x)

# Sort by chromosome number and start position
vmr_data_sorted = vmr_data.sort_values(by=['chr_num', 'start'])

# Recreate the sample values dataframe with correctly sorted VMRs
sorted_sample_values = vmr_data_sorted.iloc[:, 12:]

sorted_sample_values_transposed = sorted_sample_values.T

# Add Sample_ID to the sorted transposed data
sorted_sample_values_transposed['Sample_ID'] = sorted_sample_values_transposed.index

# Merge sorted sample values with clusters
sorted_merged_data = pd.merge(methylation_clusters, sorted_sample_values_transposed, on='Sample_ID')
sorted_merged_data = sorted_merged_data.set_index('Sample_ID')

# Split the sorted data based on eyeball_umap_clusters
sorted_cluster_1_data = sorted_merged_data[sorted_merged_data['eyeball_umap_clusters'] == 1].drop(columns=['eyeball_umap_clusters']).T
sorted_cluster_2_data = sorted_merged_data[sorted_merged_data['eyeball_umap_clusters'] == 2].drop(columns=['eyeball_umap_clusters']).T

# Adjust the index for the heatmaps
sorted_cluster_1_data.index.name = 'VMR'
sorted_cluster_2_data.index.name = 'VMR'

# Calculate the average methylation for each sample in cluster 1
average_methylation_cluster_1 = sorted_cluster_1_data.mean().sort_values(ascending=False)

# Sort the columns of sorted_cluster_1_data according to the average methylation
sorted_cluster_1_data = sorted_cluster_1_data[average_methylation_cluster_1.index]

# Calculate the average methylation for each sample in cluster 2
average_methylation_cluster_2 = sorted_cluster_2_data.mean().sort_values(ascending=False)

# Sort the columns of sorted_cluster_2_data according to the average methylation
sorted_cluster_2_data = sorted_cluster_2_data[average_methylation_cluster_2.index]

##### Layout knobs #####
RIBBON_H = 0.8 
HEATMAP_H = 2.4
FIGSIZE = (24, 16)
RIBBON_BORDERS = False

# Optionally, rename ribbon row titles here (fallback to original if missing)
row_label_map = {
    'ChemoResponse': 'Chemoresponse',
    'Met_at_Dx': 'MetDx',
    'AgeDx_Bin': 'AgeDx',
    'ATRX_expression_high_v_low': 'ATRX Expression',
    'TERT_expressed': 'TERT Expression',
    'MYC_Amplification_bin': 'MYC Amplification'
}

##### Data prep #####
combined_data = pd.concat([sorted_cluster_1_data, sorted_cluster_2_data], axis=1)
zscored_combined_data = combined_data.apply(zscore, axis=1, result_type='expand')
zscored_combined_data.columns = combined_data.columns
zscored_combined_data.index = combined_data.index
distance_matrix = hierarchy.distance.pdist(zscored_combined_data, metric='euclidean')
Z = hierarchy.linkage(distance_matrix, method='ward')

fig = plt.figure(figsize=FIGSIZE)
gs = fig.add_gridspec(
    2, 3,
    height_ratios=[RIBBON_H, HEATMAP_H],
    width_ratios=[.6, 3, 3.2],
    hspace=0.06, wspace=0.05
)

# Dendrogram axis
ax1 = fig.add_subplot(gs[1, 0])
dendro = hierarchy.dendrogram(
    Z, labels=zscored_combined_data.index, orientation='left', ax=ax1,
    leaf_font_size=12, no_labels=True, color_threshold=0,
    link_color_func=lambda k: 'black'
)
ax1.invert_yaxis()
ax1.axis('off')

# Reorder rows based on dendrogram
sorted_combined_data = zscored_combined_data.iloc[dendro['leaves'], :]
sorted_cluster_1_data_reordered = sorted_combined_data.iloc[:, :sorted_cluster_1_data.shape[1]]
sorted_cluster_2_data_reordered = sorted_combined_data.iloc[:, sorted_cluster_1_data.shape[1]:]

##### RIBBON ACROSS TOP #####
meta_cols = [
    'ChemoResponse',
    'Met_at_Dx',
    'Sex',
    'AgeDx_Bin',
    'MYC_Amplification_bin',
    'TERT_expressed',
    'ATRX_expression_high_v_low',
    'ATRX_mut',
    'TP53_mut',
    'MDM2_mut',
    'RB1_mut',
    'CDK2A_mut'
]

def _normalize_strings(s):
    return s.astype(str).str.strip().str.lower()

def get_ribbon_rgb(pheno, sample_order, meta_cols):
    ribbon_data = pheno.loc[sample_order, meta_cols].T

    def category_to_color_map(series, colname):
        # --- Explicit maps for requested fields ---
        if colname == 'ChemoResponse':
            # Try to coerce various encodings to Good/Poor
            mapped = series.map({"2": "Good", "1": "Poor"}).fillna("Missing")
            lut = { "Good": "#5ab4ac",
                    "Poor": "#d8b365",
                    "Missing": "#e5e5e5"}

        elif colname == "Met_at_Dx":
            mapped = series.map({1: "Met", 0: "No Met"}).fillna("Missing")
            lut = {
                "Met": "#af8dc3",
                "No Met": "#7fbf7b",
                "Missing": "#e5e5e5"}

        elif colname == 'Sex':
            mapped = series
            lut = {'M': '#67a9cf', 'F': '#ef8a62'}

        elif '_mut' in colname:
            mapped = series.replace({1: 'No', 2: 'Yes'})
            lut = {'Yes': "#fc8d62", 'No': '#EDEAE3'} 

        elif colname == 'AgeDx_Bin':
            mapped = series.replace({0: 'Young', 1: 'Old'})
            lut = {'Young': '#e9a3c9', 'Old': "#91cf60"} 

        elif colname in ['TERT_expressed', 'ATRX_expression_high_v_low']:
            mapped = series.replace({'1': 'Low', '2': 'High'}).fillna('Missing')
            lut = {'High': '#ffffbf', 'Low': '#91bfdb', 'Missing': '#fc8d59'}
        elif colname == 'MYC_Amplification_bin':
            mapped = series.replace({0: 'Not Amplified', 1: 'Amplified'}).fillna('Missing')
            lut = {'Not Amplified': '#998ec3', 'Amplified': '#f1a340', 'Missing': '#e5e5e5'}

        else:
            # Fallback: categorical palette for any other fields
            mapped = series
            unique_vals = sorted(pd.Series(mapped).dropna().unique(), key=lambda x: str(x))
            palette = sns.color_palette("Set2", len(unique_vals))
            lut = dict(zip(unique_vals, palette))

        return [mcolors.to_rgb(lut.get(val, (0.9, 0.9, 0.9))) for val in mapped]

    ribbon_colors = []
    for meta in ribbon_data.index:
        color_row = category_to_color_map(ribbon_data.loc[meta], meta)
        ribbon_colors.append(np.array(color_row))

    ribbon_rgb_array = np.stack(ribbon_colors, axis=0)
    return ribbon_rgb_array

##### Helper to draw thin borders on ribbon cells #####
def draw_cell_borders(ax, rgb_array, lw=0.3, color='k'):
    n_rows, n_cols = rgb_array.shape[:2]
    ax.set_axisbelow(False)
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which='minor', color=color, linestyle='-', linewidth=lw)
    ax.tick_params(which='minor', length=0)

##### HEATMAPS #####
cmap = sns.diverging_palette(250, 10, s=200, l=30, as_cmap=True)

# Cluster 1 ribbon
ax_ribbon1 = fig.add_subplot(gs[0, 1])
ribbon_rgb_1 = get_ribbon_rgb(pheno, sorted_cluster_1_data_reordered.columns, meta_cols)
ax_ribbon1.imshow(ribbon_rgb_1, aspect='auto', interpolation='none')

for side in ax_ribbon1.spines.values():
    side.set_visible(True)
    side.set_linewidth(2)     # thickness of the outer border
    side.set_edgecolor('black')

# Apply readable, bold labels w/ optional renames
yticks_1 = [row_label_map.get(m, m) for m in meta_cols]
ax_ribbon1.set_yticks(range(len(meta_cols)))
ax_ribbon1.set_yticklabels(yticks_1, fontsize=13, fontweight='bold')
ax_ribbon1.tick_params(left=True, bottom=False, pad=4)
ax_ribbon1.set_xticks([])
ax_ribbon1.set_xticklabels([])
ax_ribbon1.set_title("Global Cluster 1", fontsize=20, fontweight ="bold", fontname='Arial')

ax_ribbon1.grid(False); ax_ribbon1.set_xticks([], minor=True); ax_ribbon1.set_yticks([], minor=True)

# Cluster 1 heatmap
ax2 = fig.add_subplot(gs[1, 1])
sns.heatmap(sorted_cluster_1_data_reordered, cmap=cmap, ax=ax2, cbar=False, vmin=-3, vmax=3)
ax2.set_xticks([]); ax2.set_yticks([])
ax2.set_xlabel(''); ax2.set_ylabel('')
for spine in ax2.spines.values():
    spine.set_visible(True); spine.set_linewidth(2); spine.set_edgecolor('black')

# Cluster 2 ribbon
ax_ribbon2 = fig.add_subplot(gs[0, 2])
ribbon_rgb_2 = get_ribbon_rgb(pheno, sorted_cluster_2_data_reordered.columns, meta_cols)
ax_ribbon2.imshow(ribbon_rgb_2, aspect='auto', interpolation='none')

for side in ax_ribbon2.spines.values():
    side.set_visible(True)
    side.set_linewidth(2)
    side.set_edgecolor('black')

# Keep labels hidden on the right ribbon (redundant)
ax_ribbon2.set_yticks(range(len(meta_cols)))
ax_ribbon2.set_yticklabels([])
ax_ribbon2.set_xticks([]); ax_ribbon2.set_xticklabels([])
ax_ribbon2.tick_params(left=False, bottom=False)
ax_ribbon2.set_title("Global Cluster 2", fontsize=20, fontweight ="bold", fontname='Arial')

ax_ribbon2.grid(False); ax_ribbon2.set_xticks([], minor=True); ax_ribbon2.set_yticks([], minor=True)

# Cluster 2 heatmap
ax3 = fig.add_subplot(gs[1, 2])
sns.heatmap(sorted_cluster_2_data_reordered, cmap=cmap, ax=ax3, cbar=False, vmin=-3, vmax=3)
ax3.set_xticks([]); ax3.set_yticks([])
ax3.set_xlabel(''); ax3.set_ylabel('')
for spine in ax3.spines.values():
    spine.set_visible(True); spine.set_linewidth(2); spine.set_edgecolor('black')

plt.tight_layout()
plt.show()

### Script Name: Comparison of Methylation Pattern by MetDx

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script explores how methylations patterns within the TARGET dataset associate with metastasis status at diagnosis.

**Requires**: TARGET sample phenotype matrix and global VMR profile (M-Value)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.stats import zscore
import matplotlib.colors as mcolors

# Load the files
pheno = pd.read_csv('/Volumes/homedir$/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)
vmr_data = pd.read_csv('/Volumes/homedir$/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv', index_col=0)

# Extract relevant columns from TARGET_Methylation.csv
methylation_clusters = pheno[['eyeball_umap_clusters']]

# Ensure that the chromosome sorting is correct by handling chromosomes as strings
vmr_data['chr_num'] = vmr_data['chr'].str.extract(r'(\d+)', expand=False).fillna(vmr_data['chr'])
vmr_data['chr_num'] = vmr_data['chr_num'].apply(lambda x: int(x) if x.isdigit() else x)

# Sort by chromosome number and start position
vmr_data_sorted = vmr_data.sort_values(by=['chr_num', 'start'])

# Recreate the sample values dataframe with correctly sorted VMRs
sorted_sample_values = vmr_data_sorted.iloc[:, 12:]

sorted_sample_values_transposed = sorted_sample_values.T

# Add Sample_ID to the sorted transposed data
sorted_sample_values_transposed['Sample_ID'] = sorted_sample_values_transposed.index

# Merge sorted sample values with clusters
sorted_merged_data = pd.merge(methylation_clusters, sorted_sample_values_transposed, on='Sample_ID')
sorted_merged_data = sorted_merged_data.set_index('Sample_ID')

# Split the sorted data based on eyeball_umap_clusters
sorted_cluster_1_data = sorted_merged_data[sorted_merged_data['eyeball_umap_clusters'] == 1].drop(columns=['eyeball_umap_clusters']).T
sorted_cluster_2_data = sorted_merged_data[sorted_merged_data['eyeball_umap_clusters'] == 2].drop(columns=['eyeball_umap_clusters']).T

# Adjust the index for the heatmaps
sorted_cluster_1_data.index.name = 'VMR'
sorted_cluster_2_data.index.name = 'VMR'

# Calculate the average methylation for each sample in cluster 1
average_methylation_cluster_1 = sorted_cluster_1_data.mean().sort_values(ascending=False)

# Sort the columns of sorted_cluster_1_data according to the average methylation
sorted_cluster_1_data = sorted_cluster_1_data[average_methylation_cluster_1.index]

# Calculate the average methylation for each sample in cluster 2
average_methylation_cluster_2 = sorted_cluster_2_data.mean().sort_values(ascending=False)

# Sort the columns of sorted_cluster_2_data according to the average methylation
sorted_cluster_2_data = sorted_cluster_2_data[average_methylation_cluster_2.index]

##### Comparison of Methylation Pattern by MetDx #####

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.stats import zscore
import matplotlib.colors as mcolors

########## Layout knobs ##########
RIBBON_H = 0.1     # bigger = taller ribbons
HEATMAP_H = 2.4    # smaller = shorter heatmaps (relative to ribbons)
FIGSIZE = (24, 16) # overall figure size
RIBBON_BORDERS = False

# Optionally, rename ribbon row titles here (fallback to original if missing)
row_label_map = {
    'ChemoResponse': 'Chemoresponse',
    'Met_at_Dx': 'MetDx',
    'AgeDx_Bin': 'AgeDx',
    'ATRX_expression_high_v_low': 'ATRX Expression',
    'TERT_expressed': 'TERT Expression',
}

########## Data prep ##########
# Use the two cluster matrices only to define the full sample set
combined_data = pd.concat([sorted_cluster_1_data, sorted_cluster_2_data], axis=1)

# Z-score across rows (features)
zscored_combined_data = combined_data.apply(zscore, axis=1, result_type='expand')
zscored_combined_data.columns = combined_data.columns
zscored_combined_data.index = combined_data.index

# Split samples by Met_at_Dx (0/1) BEFORE making the gridspec
all_samples = combined_data.columns

# Met_at_Dx == 1 -> Yes
met_yes_samples = [s for s in all_samples if pheno.loc[s, 'Met_at_Dx'] in [1, '1']]

# Met_at_Dx == 0 -> No
met_no_samples  = [s for s in all_samples if pheno.loc[s, 'Met_at_Dx'] in [0, '0']]

# Sample counts
n_yes = len(met_yes_samples)
n_no  = len(met_no_samples)
total = max(1, n_yes + n_no)

# Keep dendrogram width fixed, just redistribute heatmap widths
TOTAL_HEATMAP_WIDTH = 6.2
frac_yes = n_yes / total
frac_no  = n_no / total

yes_width = TOTAL_HEATMAP_WIDTH * frac_yes
no_width  = TOTAL_HEATMAP_WIDTH * frac_no

fig = plt.figure(figsize=FIGSIZE)
gs = fig.add_gridspec(
    2, 3,
    height_ratios=[RIBBON_H, HEATMAP_H],
    width_ratios=[0.6, yes_width, no_width],
    hspace=0.06, wspace=0.05
)


# Dendrogram axis (for features/rows)
distance_matrix = hierarchy.distance.pdist(zscored_combined_data, metric='euclidean')
Z = hierarchy.linkage(distance_matrix, method='ward')

ax1 = fig.add_subplot(gs[1, 0])
dendro = hierarchy.dendrogram(
    Z, labels=zscored_combined_data.index, orientation='left', ax=ax1,
    leaf_font_size=12, no_labels=True, color_threshold=0,
    link_color_func=lambda k: 'black'
)
ax1.invert_yaxis()
ax1.axis('off')

# Reorder rows based on dendrogram
sorted_combined_data = zscored_combined_data.iloc[dendro['leaves'], :]

# Reuse the Met_at_Dx split but preserve the original column order
# (columns weren't reordered by the dendrogram, only rows were)
met_yes_data = sorted_combined_data[met_yes_samples]
met_no_data  = sorted_combined_data[met_no_samples]

########## RIBBON ACROSS TOP ##########
meta_cols = [
    'Met_at_Dx'
]

def _normalize_strings(s):
    return s.astype(str).str.strip().str.lower()

def get_ribbon_rgb(pheno, sample_order, meta_cols):
    ribbon_data = pheno.loc[sample_order, meta_cols].T

    def category_to_color_map(series, colname):
        ### Explicit maps for requested fields ###
        if colname == 'ChemoResponse':
            s = _normalize_strings(series)
            mapped = series.copy()
            mapped[(s.str.contains('good|resp|favorable', na=False)) | (series.astype(str).isin(['2']))] = 'Good'
            mapped[(s.str.contains('poor|non|unfav', na=False)) | (series.astype(str).isin(['1','0']))] = 'Poor'
            mapped = mapped.fillna('Missing')
            lut = {'Good': '#5ab4ac',
                   'Poor': '#d8b365',
                   'Missing': '#f7f7f7'}

        elif colname == 'Met_at_Dx':
            # Met_at_Dx encoded as 0/1 (or '0'/'1')
            mapped = series.replace({
                1: 'Yes', '1': 'Yes',
                0: 'No',  '0': 'No'
            })
            mapped = mapped.where(mapped.isin(['Yes', 'No']), 'Missing')
            lut = {'Yes': '#af8dc3',    # purple
                   'No':  '#7fbf7b',    # green
                   'Missing': '#f7f7f7'}

        elif colname == 'Sex':
            mapped = series
            lut = {'M': '#67a9cf', 'F': '#ef8a62'}

        elif '_mut' in colname:
            mapped = series.replace({1: 'No', 2: 'Yes'})
            lut = {'Yes': "#fc8d62", 'No': '#EDEAE3'} 

        elif colname == 'AgeDx_Bin':
            mapped = series.replace({0: 'Young', 1: 'Old'})
            lut = {'Young': '#e9a3c9', 'Old': "#91cf60"} 

        elif colname in ['TERT_expressed', 'ATRX_expression_high_v_low']:
            mapped = series.replace({'1': 'Low', '2': 'High'}).fillna('Missing')
            lut = {'High': '#ffffbf', 'Low': '#91bfdb', 'Missing': '#fc8d59'}

        else:
            # Fallback: categorical palette for any other fields
            mapped = series
            unique_vals = sorted(pd.Series(mapped).dropna().unique(), key=lambda x: str(x))
            palette = sns.color_palette("Set2", len(unique_vals))
            lut = dict(zip(unique_vals, palette))

        return [mcolors.to_rgb(lut.get(val, (0.9, 0.9, 0.9))) for val in mapped]

    ribbon_colors = []
    for meta in ribbon_data.index:
        color_row = category_to_color_map(ribbon_data.loc[meta], meta)
        ribbon_colors.append(np.array(color_row))

    ribbon_rgb_array = np.stack(ribbon_colors, axis=0)
    return ribbon_rgb_array

# Optional borders for ribbon cells
def draw_cell_borders(ax, rgb_array, lw=0.3, color='k'):
    n_rows, n_cols = rgb_array.shape[:2]
    ax.set_axisbelow(False)
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which='minor', color=color, linestyle='-', linewidth=lw)
    ax.tick_params(which='minor', length=0)

########## HEATMAPS ##########
cmap = sns.diverging_palette(250, 10, s=200, l=30, as_cmap=True)

# Panel 1: Met at Dx = Yes
ax_ribbon1 = fig.add_subplot(gs[0, 1])
ribbon_rgb_1 = get_ribbon_rgb(pheno, met_yes_samples, meta_cols)
ax_ribbon1.imshow(ribbon_rgb_1, aspect='auto', interpolation='none')

for side in ax_ribbon1.spines.values():
    side.set_visible(True)
    side.set_linewidth(2)
    side.set_edgecolor('black')

yticks_1 = [row_label_map.get(m, m) for m in meta_cols]
ax_ribbon1.set_yticks(range(len(meta_cols)))
ax_ribbon1.set_yticklabels(yticks_1, fontsize=13, fontweight='bold')
ax_ribbon1.tick_params(left=True, bottom=False, pad=4)
ax_ribbon1.set_xticks([])
ax_ribbon1.set_xticklabels([])
ax_ribbon1.set_title("Metastasis at Diagnosis: Yes", fontsize=20, fontweight="bold", fontname='Arial')

ax_ribbon1.grid(False); ax_ribbon1.set_xticks([], minor=True); ax_ribbon1.set_yticks([], minor=True)

ax2 = fig.add_subplot(gs[1, 1])
sns.heatmap(met_yes_data, cmap=cmap, ax=ax2, cbar=False, vmin=-3, vmax=3)
ax2.set_xticks([]); ax2.set_yticks([])
ax2.set_xlabel(''); ax2.set_ylabel('')
for spine in ax2.spines.values():
    spine.set_visible(True); spine.set_linewidth(2); spine.set_edgecolor('black')

# Panel 2: Met at Dx = No 
ax_ribbon2 = fig.add_subplot(gs[0, 2])
ribbon_rgb_2 = get_ribbon_rgb(pheno, met_no_samples, meta_cols)
ax_ribbon2.imshow(ribbon_rgb_2, aspect='auto', interpolation='none')

for side in ax_ribbon2.spines.values():
    side.set_visible(True)
    side.set_linewidth(2)
    side.set_edgecolor('black')

ax_ribbon2.set_yticks(range(len(meta_cols)))
ax_ribbon2.set_yticklabels([])
ax_ribbon2.set_xticks([]); ax_ribbon2.set_xticklabels([])
ax_ribbon2.tick_params(left=False, bottom=False)
ax_ribbon2.set_title("Metastasis at Diagnosis: No", fontsize=20, fontweight="bold", fontname='Arial')

ax_ribbon2.grid(False); ax_ribbon2.set_xticks([], minor=True); ax_ribbon2.set_yticks([], minor=True)

ax3 = fig.add_subplot(gs[1, 2])
sns.heatmap(met_no_data, cmap=cmap, ax=ax3, cbar=False, vmin=-3, vmax=3)
ax3.set_xticks([]); ax3.set_yticks([])
ax3.set_xlabel(''); ax3.set_ylabel('')
for spine in ax3.spines.values():
    spine.set_visible(True); spine.set_linewidth(2); spine.set_edgecolor('black')

plt.tight_layout()
plt.show()


In [ ]:
##### Comparison of Average Methylation by MetDx #####

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns

##### Average methylation per sample #####
met_pos_samples = pheno.index[pheno['Met_at_Dx'] == 1]
met_neg_samples = pheno.index[pheno['Met_at_Dx'] == 0]

avg_meth_pos = combined_data[met_pos_samples].mean(axis=0)
avg_meth_neg = combined_data[met_neg_samples].mean(axis=0)

##### Build tidy dataframe for plotting #####
df_plot = pd.DataFrame({
    'Sample': list(avg_meth_pos.index) + list(avg_meth_neg.index),
    'Average_Methylation': list(avg_meth_pos.values) + list(avg_meth_neg.values),
    'Metastasis': ['Met+'] * len(avg_meth_pos) + ['Met-'] * len(avg_meth_neg)
})

##### Statistics #####
# Welch's t-test
t_stat, p_val = ttest_ind(avg_meth_pos, avg_meth_neg, equal_var=False)

# Cohen's d
def cohens_d(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    pooled_sd = np.sqrt(((nx - 1)*vx + (ny - 1)*vy) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_sd if pooled_sd > 0 else np.nan

d = cohens_d(avg_meth_pos, avg_meth_neg)

# **Mean difference**
mean_pos = avg_meth_pos.mean()
mean_neg = avg_meth_neg.mean()
mean_diff = mean_pos - mean_neg

print("Average methylation per sample (Met+ vs Met-):")
print(f"  Met+ mean ± SD: {mean_pos:.4f} ± {avg_meth_pos.std(ddof=1):.4f}")
print(f"  Met- mean ± SD: {mean_neg:.4f} ± {avg_meth_neg.std(ddof=1):.4f}")
print(f"\nMean difference (Met+ - Met-): {mean_diff:.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4e}")
print(f"Cohen's d: {d:.4f}")

##### Boxplot with jitter #####
plt.figure(figsize=(6, 6))

sns.boxplot(
    data=df_plot,
    x="Metastasis",
    y="Average_Methylation",
    width=0.5,
    palette=["#af8dc3", "#7fbf7b"]
)

sns.stripplot(
    data=df_plot,
    x="Metastasis",
    y="Average_Methylation",
    color="black",
    size=6,
    jitter=0.15,
    alpha=0.7
)

plt.title("Average Methylation by Metastasis at Diagnosis", fontsize=16)
plt.xlabel("")
plt.ylabel("Average Methylation (mean across VMRs)", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
##### Univariable VMR Analysis by MetDx #####

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

results = []

for vmr in combined_data.index:
    x = combined_data.loc[vmr, met_pos_samples].astype(float)
    y = combined_data.loc[vmr, met_neg_samples].astype(float)

    x = x.dropna()
    y = y.dropna()

    if len(x) < 3 or len(y) < 3:
        continue

    # Welch's t-test
    t_stat, p_val = ttest_ind(x, y, equal_var=False)

    # Cohen's d
    nx, ny = len(x), len(y)
    vx, vy = x.var(ddof=1), y.var(ddof=1)
    pooled_sd = np.sqrt(((nx - 1)*vx + (ny - 1)*vy) / (nx + ny - 2))
    d = (x.mean() - y.mean()) / pooled_sd if pooled_sd > 0 else np.nan

    results.append({
        'VMR': vmr,
        'mean_met_pos': x.mean(),
        'mean_met_neg': y.mean(),
        'mean_diff': x.mean() - y.mean(),
        't_stat': t_stat,
        'p_value': p_val,
        'cohens_d': d
    })

results_df = pd.DataFrame(results)
print("Univariable Analysis of VMRs by MetDx")
print(f"Tested {results_df.shape[0]} VMRs")

reject, qvals, _, _ = multipletests(
    results_df['p_value'],
    alpha=0.05,
    method='fdr_bh'
)

results_df['q_value'] = qvals
results_df['significant'] = reject

print("Min p-value:", results_df['p_value'].min())
print("Significant VMRs (p < 0.05):", len(results_df[results_df['p_value'] <= 0.05]))
print("Significant VMRs (FDR < 0.05):", results_df['significant'].sum())


### Script Name: Counfounder Association to Unsupervised Global Clustering

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script calculates assosiation between unsupervised global VMR clustering and several potential confounders. These include Chemoresponse, Metastasis at Diagnosis (MetDx), Sex, Age at Diagnosis (AgeDx) - both continuous and binarized, Batch, Expression of Tert and ATRX, and mutations of ATRX, TP53, MDM2, and RB1.

**Requires**: TARGET sample phenotype matrix 

In [ ]:
# dependencies
import pandas as pd
from scipy.stats import ttest_ind, chi2_contingency

pheno = pd.read_csv('/Volumes/homedir$/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)
global_clust = pheno['eyeball_umap_clusters']
global_group1 = global_clust[global_clust == 1]
global_group2 = global_clust[global_clust == 2]
# Potential Confounders
cr = pheno['ChemoResponse']
met = pheno['Met_at_Dx']
sex = pheno['Sex']
age_dx = pheno['AgeDx'] # actual age at diagnosis in years
age_bin = pheno['AgeDx_Bin'] # age aplit at the median
batch = pheno['batch'] 
tert_expr_bin = pheno['TERT_expressed'] # No expr or some expr
# tert_expr_trinary = pheno['TERT_no_exp_low_exp_and_high_exp'] # no, low, or high expression
ATRX_expr_bin = pheno['ATRX_expression_high_v_low'] # expression split at the (mean?)

# Mutations
ATRX_mut = pheno['ATRX_mut'] # mutation of the ATRX oncogene
TP53_mut = pheno['TP53_mut'] # mutation of the TP553 oncogene
MDM2_mut = pheno['MDM2_mut'] # mutation of the MDM2 oncogene
RB1_mut = pheno['RB1_mut'] # mutation of the RB1 oncogene
CDK2A_mut = pheno['CDK2A_mut'] # mutation of the CDK2A oncogene

results = []

# Metastasis
cont_table_met = pd.crosstab(global_clust, met)
print(cont_table_met)
chi2_met, p_met, _, _ = chi2_contingency(cont_table_met)
print(f'Chi2 Statistics for Age (Bin) = {chi2_met:.4f}, p={p_met:.4f}\n')
results.append({
    "Variable": "Metastasis",
    "Test": "Chi2",
    "Statistic": chi2_met,
    "p-value": p_met
})

# Age (Continuous)
age1 = age_dx[global_group1.index]
age2 = age_dx[global_group2.index]
t_stat_age, p_age_cont = ttest_ind(age1, age2)
print(f'T-Statistic for Age (Cont) = {t_stat_age:.4f}, p={p_age_cont:.4f}\n')
results.append({
    "Variable": "Age (Cont)",
    "Test": "t-test",
    "Statistic": t_stat_age,
    "p-value": p_age_cont
})


# Age (Binary)
cont_table_age = pd.crosstab(global_clust, age_bin)
print(cont_table_age)
chi2_age, p_age_bin, _, _ = chi2_contingency(cont_table_age)
print(f'Chi2 Statistics for Age (Bin) = {chi2_age:.4f}, p={p_age_bin:.4f}\n')
results.append({
    "Variable": "Age (Bin)",
    "Test": "Chi2",
    "Statistic": chi2_age,
    "p-value": p_age_bin
})


# Sex
cont_table_sex = pd.crosstab(global_clust, sex)
print(cont_table_sex)
chi2_sex, p_sex, _, _ = chi2_contingency(cont_table_sex)
print(f'Chi2 Statistics for Age (Bin) = {chi2_sex:.4f}, p={p_sex:.4f}\n')
results.append({
    "Variable": "Sex",
    "Test": "Chi2",
    "Statistic": chi2_sex,
    "p-value": p_sex
})


# Batch (Quaternary)
cont_table_batch = pd.crosstab(global_clust, batch)
print(cont_table_batch)
chi2_batch, p_batch_bin, _, _ = chi2_contingency(cont_table_batch)
print(f'Chi2 Statistics for Batch = {chi2_batch:.4f}, p={p_batch_bin:.4f}\n')
results.append({
    "Variable": "Batch",
    "Test": "Chi2",
    "Statistic": chi2_batch,
    "p-value": p_batch_bin
})


# TERT Expression (Binary)
# remove missing values
tert_expr_bin_temp = tert_expr_bin[(tert_expr_bin != " ")] # Should remove PAPVYW sample

cont_table_tert_expr_bin = pd.crosstab(global_clust, tert_expr_bin_temp)
print(cont_table_tert_expr_bin)
chi2_tert_expr, p_tert_expr_bin, _, _ = chi2_contingency(cont_table_tert_expr_bin)
print(f'Chi2 Statistics for tert_expr_bin = {chi2_tert_expr:.4f}, p={p_tert_expr_bin:.4f}\n')
results.append({
    "Variable": "TERT Expr",
    "Test": "Chi2",
    "Statistic": chi2_tert_expr,
    "p-value": p_tert_expr_bin
})


# ATRX Expression (Binary)
ATRX_expr_bin_temp = ATRX_expr_bin[ ATRX_expr_bin != " "] # Should remove PAPVYW sample
cont_table_ATRX_expr_bin = pd.crosstab(global_clust, ATRX_expr_bin_temp)
print(cont_table_ATRX_expr_bin)
chi2_ATRX_expr, p_ATRX_expr_bin, _, _ = chi2_contingency(cont_table_ATRX_expr_bin)
print(f'Chi2 Statistics for tert_expr_bin = {chi2_ATRX_expr:.4f}, p={p_ATRX_expr_bin:.4f}\n')
results.append({
    "Variable": "ATRX Expr",
    "Test": "Chi2",
    "Statistic": chi2_ATRX_expr,
    "p-value": p_ATRX_expr_bin
})


# ATRX Mutation (Binary)
cont_table_ATRX_mut = pd.crosstab(global_clust, ATRX_mut)
print(cont_table_ATRX_mut)
chi2_ATRX_mut, p_ATRX_mut, _, _ = chi2_contingency(cont_table_ATRX_mut)
print(f'Chi2 Statistics for tert_expr_bin = {chi2_ATRX_mut:.4f}, p={p_ATRX_mut:.4f}\n')
results.append({
    "Variable": "ATRX Mutation",
    "Test": "Chi2",
    "Statistic": chi2_ATRX_mut,
    "p-value": p_ATRX_mut
})


# TP53 Mutation (Binary)
cont_table_TP53_mut = pd.crosstab(global_clust, TP53_mut)
print(cont_table_TP53_mut)
chi2_TP53_mut, p_TP53_mut, _, _ = chi2_contingency(cont_table_TP53_mut)
print(f'Chi2 Statistics for tert_expr_bin = {chi2_TP53_mut:.4f}, p={p_TP53_mut:.4f}\n')
results.append({
    "Variable": "TP53 Mutation",
    "Test": "Chi2",
    "Statistic": chi2_TP53_mut,
    "p-value": p_TP53_mut
})


# MDM2 Mutation (Binary)
cont_table_MDM2_mut = pd.crosstab(global_clust, MDM2_mut)
print(cont_table_MDM2_mut)
chi2_MDM2_mut, p_MDM2_mut, _, _ = chi2_contingency(cont_table_MDM2_mut)
print(f'Chi2 Statistics for tert_expr_bin = {chi2_MDM2_mut:.4f}, p={p_MDM2_mut:.4f}\n')
results.append({
    "Variable": "MDM2 Mutation",
    "Test": "Chi2",
    "Statistic": chi2_MDM2_mut,
    "p-value": p_MDM2_mut
})


# RB1 Mutation (Binary)
cont_table_RB1_mut = pd.crosstab(global_clust, RB1_mut)
print(cont_table_RB1_mut)
chi2_RB1_mut, p_RB1_mut, _, _ = chi2_contingency(cont_table_RB1_mut)
print(f'Chi2 Statistics for tert_expr_bin = {chi2_RB1_mut:.4f}, p={p_RB1_mut:.4f}\n')
results.append({
    "Variable": "RB1 Mutation",
    "Test": "Chi2",
    "Statistic": chi2_RB1_mut,
    "p-value": p_RB1_mut
})


# CDK2A Mutation (Binary)
cont_table_CDK2A_mut = pd.crosstab(global_clust, CDK2A_mut)
print(cont_table_CDK2A_mut)
chi2_CDK2A_mut, p_CDK2A_mut, _, _ = chi2_contingency(cont_table_CDK2A_mut)
print(f'Chi2 Statistics for tert_expr_bin = {chi2_CDK2A_mut:.4f}, p={p_CDK2A_mut:.4f}\n')
results.append({
    "Variable": "CDK2A Mutation",
    "Test": "Chi2",
    "Statistic": chi2_CDK2A_mut,
    "p-value": p_CDK2A_mut
})

results_df = pd.DataFrame(results)
results_df = results_df[["Variable", "Test", "Statistic", "p-value"]]  # reorder columns
results_df["p-value"] = results_df["p-value"].map("{:.4f}".format)
results_df["Statistic"] = results_df["Statistic"].map("{:.4f}".format)

# Optionally, save results
# results_df.to_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/confounding_with_global_clusters.csv', index=False)

### Script Name: Survival Analysis (Unsupervised)

**Authors**: Joshua Bowers

**Language**: Python

**Description**: Kaplan-Meier survival analsyis for unstratified and metastasis-stratified unsupervised clustering results

**Requires**: TARGET sample phenotype matrix

In [ ]:
##### Survival Analysis of Unsupervised UMAP Clusters ##### 

## User Input ##

survival_months = 'RFS_months'  # 'RFS_months' or 'Surv_months'
survival_cens   = 'RFS_cens'    # 'RFS_cens'   or 'Surv_cens'

###############

# Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test
import itertools

# read in phenotype data for samples
pheno = pd.read_csv('/Volumes/homedir$/Projects/Publication_Materials/Manuscript_Pheno.csv', index_col=0)

# Initialize the Kaplan-Meier fitter
kmf = KaplanMeierFitter()

# Plot setup
plt.figure(figsize=(10, 7))

for group in [1, 2]:
    # Subset to the group
    group_data = pheno[pheno['global_umap_clust'] == group]
    
    # Fit the Kaplan-Meier estimator
    kmf.fit(durations=group_data[survival_months], event_observed=group_data[survival_cens], label=f'Cluster {group}')
    
    # Plot the survival function
    kmf.plot_survival_function()

    # Get and print the median survival time
    median_survival = kmf.median_survival_time_
    print(f'Median survival time for cluster {group}: {median_survival} months')

#plt.title('')
plt.xlabel(' ', fontsize=16, fontweight='bold')
#plt.ylabel('Overall Survival Probability', fontsize=16, fontweight='bold')
plt.grid(False)
plt.yticks(fontsize=16, fontweight='bold')
plt.xticks(fontsize=16, fontweight='bold')  
ax = plt.gca()
ax.set_ylim(-0.05, 1.05)
plt.legend(fontsize=14)
#plt.legend([],[], frameon=False)
plt.show()

results = logrank_test(
    durations_A=pheno[pheno['global_umap_clust'] == 1]['RFS_months'],
    durations_B=pheno[pheno['global_umap_clust'] == 2]['RFS_months'],
    event_observed_A=pheno[pheno['global_umap_clust'] == 1]['RFS_cens'],
    event_observed_B=pheno[pheno['global_umap_clust'] == 2]['RFS_cens']
)
print('Log-rank Test p-value:', results.p_value)

# -------- Binary HR (Cox model) between Cluster 1 and Cluster 2 --------
# Make a small DF for the Cox model
cox_df = pheno[['RFS_months', 'RFS_cens', 'global_umap_clust']].dropna().copy()

# Recode clusters to 0/1 so Cluster 1 is reference (0) and Cluster 2 is comparison (1)
cox_df['cluster_bin'] = np.where(cox_df['global_umap_clust'] == 2, 1, 0)

# Fit Cox proportional hazards model
cph = CoxPHFitter()
cph.fit(
    cox_df[['RFS_months', 'RFS_cens', 'cluster_bin']],
    duration_col='RFS_months',
    event_col='RFS_cens'   # make sure 1 = event, 0 = censored
)

# Extract HR and CI for the binary cluster variable
summary = cph.summary.loc['cluster_bin']
hr = summary['exp(coef)']
ci_low = summary['exp(coef) lower 95%']
ci_high = summary['exp(coef) upper 95%']
p_hr = summary['p']

hr_flipped = 1 / hr
ci_low_flipped = 1 / ci_high
ci_high_flipped = 1 / ci_low

print(f"HR (Cluster 2 vs Cluster 1): {hr:.2f} "
      f"(95% CI: {ci_low:.2f}–{ci_high:.2f}), p = {p_hr:.3g}")

print(f"HR (Cluster 1 vs Cluster 2): {hr_flipped:.2f} "
      f"(95% CI: {ci_low_flipped:.2f}–{ci_high_flipped:.2f}), p = {p_hr:.3g}")


In [ ]:
##### Metastasis Stratified Survival Analysis of Unsupervised UMAP Clusters ##### 

## User Input ##

survival_months = 'RFS_months'  # 'RFS_months' or 'Surv_months'
survival_cens   = 'RFS_cens'    # 'RFS_cens'   or 'Surv_cens'

###############

# Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test
import itertools

# read in phenotype data for samples
pheno = pd.read_csv('/Volumes/homedir$/Projects/Publication_Materials/Manuscript_Pheno.csv', index_col=0)


# build anlaysis dataframe
surv_df = pheno[[survival_months, survival_cens, "Met_at_Dx", "global_umap_clust"]].copy()
surv_df = surv_df.dropna()

# Define grouping
surv_df["risk_group"] = surv_df["global_umap_clust"].map({1: "High", 2: "Low"})
surv_df = surv_df.dropna(subset=["risk_group"])

# KM plotting
cluster_map = {"High": "High Risk", "Low": "Low Risk"}
color_map   = {"High Risk": "C0", "Low Risk": "C1"}

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(10, 7))

for group in ['High', 'Low']:
    for met_at_dx in sorted(surv_df['Met_at_Dx'].unique()):
        subset = surv_df[(surv_df['risk_group'] == group) & (surv_df['Met_at_Dx'] == met_at_dx)]
        if subset.empty:
            continue

        kmf.fit(durations=subset[survival_months], event_observed=subset[survival_cens])

        cluster_label = cluster_map[group]
        color = color_map[cluster_label]
        ls    = '--' if met_at_dx == 1 else '-'   # dashed for Met positive

        kmf.plot_survival_function(
            ax=ax,
            color=color,
            linestyle=ls,
            linewidth=2,
            ci_show=False,
            label=f'{cluster_label}, Met {"Positive" if met_at_dx == 1 else "Negative"}'
        )

        # Get and print the median survival time
        median_survival = kmf.median_survival_time_
        print(f'Median survival time for {group} risk, met {"positive" if met_at_dx == 1 else "negative"}: {median_survival} months')

# Aesthetics and labeling
ax.set_xlabel(' ', fontsize=16, fontweight='bold')
#ax.set_ylabel('OS Probability', fontsize=16, fontweight='bold')
ax.grid(False)
plt.yticks(fontsize=16, fontweight='bold')
plt.xticks(fontsize=16, fontweight='bold')
ax = plt.gca()
ax.set_ylim(-0.05, 1.05)
plt.legend(fontsize=14)

plt.show()

##### Multigroup log-rank across the 4 strata #####
# Note: This is not the same as stratified log-rank
surv_df['combined_group'] = (
    surv_df['risk_group'].map(cluster_map) +
    ', Met ' + surv_df['Met_at_Dx'].map({0: 'Negative', 1: 'Positive'})
)

multivariate_results = multivariate_logrank_test(
    event_durations=surv_df[survival_months],
    groups=surv_df['combined_group'],
    event_observed=surv_df[survival_cens]
)
print('Multivariate Log-rank Test p-value:')
print(multivariate_results.summary)

##### Pairwise log-rank tests across the 4 strata #####

# Make clean labels for the 4 combinations
surv_df['pair_group'] = (
    surv_df['risk_group'].map(cluster_map) +
    ', Met ' + surv_df['Met_at_Dx'].map({0: 'Negative', 1: 'Positive'})
)

groups = surv_df['pair_group'].unique()
groups = sorted(groups)  # nice ordering

print("\nPAIRWISE LOG-RANK TESTS:\n")

for g1, g2 in itertools.combinations(groups, 2):
    df1 = surv_df[surv_df['pair_group'] == g1]
    df2 = surv_df[surv_df['pair_group'] == g2]

    result = logrank_test(
        df1[survival_months], df2[survival_months],
        df1[survival_cens],   df2[survival_cens]
    )

    print(f"{g1}  vs  {g2}")
    print(f"p = {result.p_value:.4g}")
    print("----------------------------------")

##### Pairwise Cox models with HR flipped so higher-risk is on top #####

print("\nPAIRWISE COX HAZARD RATIOS (higher-risk vs lower-risk):\n")

cph = CoxPHFitter()

for g1, g2 in itertools.combinations(groups, 2):
    df_pair = surv_df[surv_df['pair_group'].isin([g1, g2])].copy()
    # Binary indicator: by default, g2 = 1, g1 = 0
    df_pair['group_bin'] = (df_pair['pair_group'] == g2).astype(int)

    # Fit Cox model
    cph.fit(
        df_pair[[survival_months, survival_cens, 'group_bin']],
        duration_col=survival_months,
        event_col=survival_cens
    )

    s = cph.summary.loc['group_bin']
    hr      = s['exp(coef)']
    ci_low  = s['exp(coef) lower 95%']
    ci_high = s['exp(coef) upper 95%']
    p       = s['p']

    # Flip if HR < 1 so the reported HR is always > 1
    # and corresponds to "higher hazard vs lower hazard"
    if hr >= 1:
        ref_group  = g1
        comp_group = g2
        hr_out, ci_low_out, ci_high_out = hr, ci_low, ci_high
    else:
        # invert HR and CI; swap group labels
        ref_group  = g2
        comp_group = g1
        hr_out     = 1.0 / hr
        ci_low_out = 1.0 / ci_high
        ci_high_out = 1.0 / ci_low

    print(f"{comp_group}  vs  {ref_group}")
    print(f"HR = {hr_out:.2f} (95% CI {ci_low_out:.2f}–{ci_high_out:.2f}), p = {p:.3g}")
    print("----------------------------------")


## Section 4: Supervised Analysis

### Script Name: Univariable VMR Survival Association Analysis

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script defines which VMRs of the global profile individually associate with survival using Cox Proportional Hazard (CPH) modeling.

**Requires**: TARGET sample phenotype matrix and global VMR profile (M-values)

In [ ]:
##### Non-Corrected Univariable Cox Regression Analysis #####

### User Input: ####

# RFS or OS?
survival_time = 'RFS_months'
survival_cens = 'RFS_cens'

# define stringency of filtering. A stringency of 0.1 will keep univariably significant values with a corrected p-value less than 0.1
P_stringency = 0.1

####################

# Dependencies
import pandas as pd
from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests

# read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

# read in original methylation values
result_df = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv')

# Extract the columns representing average values for samples
vmr_vals = result_df.iloc[:, 11:].T.astype(float)

# Set column names to be actual VMR descriptions
vmr_names = result_df['chr'].astype(str) + "-" + result_df['start'].astype(str) + "-" + result_df['end'].astype(str)
vmr_vals.columns = vmr_names

# Merge phenotype and VMR values
merged_df = pd.merge(vmr_vals, pheno[[survival_time, survival_cens]], left_index=True, right_index=True)

# List of features
features = list(vmr_vals.columns)

cph = CoxPHFitter()
summary_stats_fragments = []

for feature in features:
    try:
        # DataFrame for Cox regression (feature only, not Met_at_Dx)
        cox_df = merged_df[[feature, survival_time, survival_cens]]

        # Fit Cox model (univariable: feature only)
        cph.fit(cox_df, duration_col=survival_time, event_col=survival_cens, formula=f'`{feature}`')

        # Extract summary for the feature
        feature_summary = cph.summary.loc[[feature]].copy()
        feature_summary['feature'] = feature 
        summary_stats_fragments.append(feature_summary)
    except Exception as e:
        print(f"Could not fit a Cox model for {feature}. Reason: {e}")

# Concatenate results into a DataFrame
cox_stats_df = pd.concat(summary_stats_fragments, axis=0)

# FDR correction
_, p_corrected, _, _ = multipletests(cox_stats_df['p'].values, alpha=P_stringency, method='fdr_bh')
cox_stats_df['p_corrected'] = p_corrected

cox_stats_df[cox_stats_df['p_corrected'] < P_stringency]

### Script Name: Supervised Risk Prediction

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script creates a signed average supervised prediction model to define expected risk and long-term survival using univariably-associated TARGET VMRs. It utilizes leave-one-out cross validation (LOOCV) coupled with time-dependent ROC analysis as a means of internal performance assessment.

**Requires**: TARGET sample phenotype matrix and global VMR profile (M-values)

In [ ]:
##### User Input #####

# RFS or OS?
survival_time = 'Surv_months'
survival_cens = 'Surv_cens'

# Correct for covariates? 
# None if no correction | 'Met_at_Dx' if correcting for Met_at_Dx | 'CR' if correcting for chemoresponse
corrected = None

# Stratified for metastasis? - REMINDER: Cannot be both stratified and corrected
stratified = False

# define stringency of filtering. A stringency of 0.1 will keep univariably significant values with a corrected p-value less than 0.1
P_stringency = 0.1

####################

# Dependencies
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests

# load Data
# read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

# read in original methylation values
result_df = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv')

# Extract the columns representing average values for samples
vmr_vals = result_df.iloc[:, 11:].T.astype(float)

# Set column names to be actual VMR descriptions
vmr_names = result_df['chr'].astype(str) + "-" + result_df['start'].astype(str) + "-" + result_df['end'].astype(str)
vmr_vals.columns = vmr_names

# Compile survival and methylation results
merged_df = pd.merge(vmr_vals, pheno[[survival_time, survival_cens]], left_index=True, right_index=True)

# Initialize cross validation loop
loo = LeaveOneOut()

# Performance metric
risk_scores = np.zeros(len(merged_df))
remaining_VMRs = np.zeros(len(merged_df))

# Create list of features
features = list(vmr_vals.columns)

# DataFrame to store coefficients and p-values
coef_summary_df = pd.DataFrame()

# Start of Leave-One-Out Cross-Validation loop
for train_index, test_index in loo.split(merged_df):
    # Initialize the Cox model for every cross validation
    cph = CoxPHFitter()
    train_df = merged_df.iloc[train_index].copy()
    test_df = merged_df.iloc[test_index].copy()

    # Univariate Cox regression
    summary_stats_fragments = [] 

    for feature in features:
        try:
            # Create a df specifically for cox regression - needs survival_cens and survival_time inside the same df
            cox_df = train_df[[feature, survival_time, survival_cens]]

            # Fit the model
            cph.fit(cox_df, duration_col=survival_time, event_col=survival_cens)

            # Create summary 
            feature_summary = cph.summary.loc[[feature]].copy()
            feature_summary['feature'] = feature 
            summary_stats_fragments.append(feature_summary)
        except Exception as e:
            # Handle cases where the model might not converge or other issues
            print(f"Could not fit a Cox model for {feature}. Reason: {e}")

    cox_stats_df = pd.concat(summary_stats_fragments, axis=0)

    # FDR correction
    _, p_corrected, _, _ = multipletests(cox_stats_df['p'].values, alpha=P_stringency, method='fdr_bh')

    # Save FDR stats
    cox_stats_df['p_corrected'] = p_corrected
    
    # remove elements from cox_stats_df with pvalue lower than specified stringency
    cox_stats_df = cox_stats_df[cox_stats_df['p_corrected'] < P_stringency]
    print(f'{len(cox_stats_df)} significant VMRs found when removing {merged_df.index[test_index[0]]}')
    remaining_VMRs[test_index] = len(cox_stats_df)

    if not cox_stats_df.empty:
        # Store coefficients and p-values
        cox_stats_df['train_index'] = [train_index] * len(cox_stats_df)
        coef_summary_df = pd.concat([coef_summary_df, cox_stats_df], axis=0)

        # Apply signs
        sign_mapping = cox_stats_df['coef'].apply(np.sign).to_dict()
        selected_features = cox_stats_df['feature'][cox_stats_df['p_corrected'] < P_stringency].tolist()

        # Modify train and test DataFrames in place
        for feature in selected_features:
            train_df.loc[:, feature] *= sign_mapping[feature]
            test_df.loc[:, feature] *= sign_mapping[feature]

        # Calculate signed average for both training and test set
        sa_train = train_df[selected_features].mean(axis=1)
        sa_test = test_df[selected_features].mean(axis=1)

        # Ensure sa_test is a scalar if it's not already
        sa_test_value = sa_test.iloc[0] if isinstance(sa_test, pd.Series) else sa_test

        # Fit Cox model on training data signed average
        sa_train_df = pd.DataFrame({'SignedAvg': sa_train,
                                    survival_time: train_df[survival_time],
                                    survival_cens: train_df[survival_cens]})
        if stratified == True or corrected == 'Met_at_Dx':
            sa_train_df = pd.merge(sa_train_df, pheno['Met_at_Dx'], left_index=True, right_index=True)
        if corrected == 'CR':
            sa_train_df = pd.merge(sa_train_df, pheno['ChemoResponse'], left_index=True, right_index=True)
        
        if stratified == True:
            cph.fit(sa_train_df, duration_col=survival_time, event_col=survival_cens, strata='Met_at_Dx')
        elif corrected == 'Met_at_Dx':
            cph.fit(sa_train_df, duration_col=survival_time, event_col=survival_cens, formula='SignedAvg + Met_at_Dx')
        elif corrected == 'CR':
            cph.fit(sa_train_df, duration_col=survival_time, event_col=survival_cens, formula='SignedAvg + ChemoResponse')
        else:
            cph.fit(sa_train_df, duration_col=survival_time, event_col=survival_cens)

        # Predict risk score for the test sample using the scalar value
        sa_test_df = pd.DataFrame({'SignedAvg': [sa_test_value]}, index=[test_df.index[0]])
        if stratified == True or corrected == 'Met_at_Dx':
            sa_test_df = pd.merge(sa_test_df, pheno['Met_at_Dx'], left_index=True, right_index=True)
        if corrected == "CR":
            sa_test_df = pd.merge(sa_test_df, pheno['ChemoResponse'], left_index=True, right_index=True)
        
        risk_scores[test_index] = cph.predict_partial_hazard(sa_test_df).values
    else:
        risk_scores[test_index] = np.nan
    
# Output the risk scores
print(risk_scores)

# Output coefficient and p-value summary
print(coef_summary_df[['coef', 'p', 'p_corrected']])

risk_df = pd.DataFrame({
    'sample': merged_df.index, 
    'risk_scores': risk_scores,
    'RFS_months':merged_df[survival_time],
    'RFS_cens': merged_df[survival_cens]})

if corrected=='Met_at_Dx':
    risk_df = pd.merge(risk_df, pheno['Met_at_Dx'], left_on='sample', right_index=True)

# Optionally, save risk scores to df
# risk_df.to_csv('/Projects/Methylation_Project/TARGET_Continued/signed_avg/risk_scores/risk_scores_FDR1_OS.csv', index=False)

In [ ]:
##### Time-Dependent ROC Curve #####

### User Input ###

# RFS or OS?
survival_time = 'RFS_months' # Surv_months = OS, RFS_months = RFS
survival_cens = 'RFS_cens' # Surv_cens = OS, RFS_cens = RFS

# Evaluate at x months
eval_time = 60 # Example: 12 months (1 year) and 60 months (5 years)

####################

# Dependencies
import pandas as pd
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc

# read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

# read in risk scores - Should have saved from previous section
if 'risk_df' not in globals():
    risk_df = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/signed_avg/risk_scores/risk_scores_FDR1.csv', index_col=0)

# Structured survival array
y_surv = Surv.from_arrays(
    event=pheno.loc[risk_df.index, survival_cens].values.astype(bool),
    time=pheno.loc[risk_df.index, survival_time].values
)

# Compute time-dependent AUC
_, auc = cumulative_dynamic_auc(
    survival_train=y_surv,
    survival_test=y_surv,
    estimate=risk_df['risk_scores'],
    times=[eval_time]
)

print(f"\nTime-dependent AUC at 5 years (60 months): {auc:.3f}")

### Script Name: Univariable VMR Chemoresponse Association Analysis

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script defines which VMRs of the global profile individually associate with response to chemotherapy (chemoresponse) using standard logistic regression modeling.

**Requires**: TARGET sample phenotype matrix and global VMR profile (M-values)

In [ ]:
# Dependencies
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2_contingency, fisher_exact
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.contingency_tables import Table2x2

# read in data
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)
chemo_data = pheno['ChemoResponse']

vmr_set = pd.read_excel('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/sampleAvg_ONLY_vmr_using_dmrcate_beta_lam500.xlsx', index_col=0).T

# Merge chemoresponse data and VMR data
merged_df = pd.merge(vmr_set, chemo_data, left_index=True, right_index=True)

# remove all samples with no chemoresponse data
merged_df = merged_df[merged_df['ChemoResponse'] != " "]
merged_df['ChemoResponse'] = merged_df['ChemoResponse'].astype(int)

# Subtract 1 from each value in the ChemoResponse column to matched requirements for statsmodels's logistic regression algorithm
merged_df['ChemoResponse'] = merged_df['ChemoResponse'] - 1

# Assume merged_df is your DataFrame
response = merged_df['ChemoResponse']
predictors = merged_df.drop('ChemoResponse', axis=1)

results_summary = []
pvalues = [] 

for column in predictors:
    X = sm.add_constant(predictors[column])  # Add a constant term for the intercept
    model = sm.Logit(response, X)
    result = model.fit(disp=0)  # disp=0 suppresses the output during the fitting process
    pvalues.append(result.pvalues[column])  # Collect the p-value for FDR correction
    summary = {
        'Predictor': column,
        'Coefficient': result.params[column],
        'P-value': result.pvalues[column],
        'Odds Ratio': np.exp(result.params[column])
    }
    results_summary.append(summary)

# Apply FDR correction
reject, pvals_corrected, _, _ = multipletests(pvalues, alpha=0.05, method='fdr_bh')

# Add corrected p-values and rejection decision to the summary
for i, summary in enumerate(results_summary):
    summary['Corrected P-value'] = pvals_corrected[i]
    summary['Significant'] = True if reject[i] else False

# Convert summary to DataFrame for easier viewing/manipulation
results_df = pd.DataFrame(results_summary)

### Script Name: Supervised Chemoresponse Prediction

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script creates a supervised prediction model to define expected chemoresponse using univariably-associated TARGET VMRs. It utilizes leave-one-out cross validation (LOOCV) coupled with ROC analysis as a means of internal performance assessment (other performance metrics were also recorded). This script supports the implementation of random forest, lasso-penalized logistic regression, signed average (code cell #2) and more.

**Requires**: TARGET sample phenotype matrix and global VMR profile (M-values)

In [ ]:
#### Supervised Chemoresponse Prediction #####

# Dependencies
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve, auc
import time
import matplotlib.pyplot as plt

# what model do you want to run in this script?
model_name = 'lasso_LR'

# function to define the model fitting and prediction function
def fit_predict_model(model_name, X_train, y_train, X_test):
    if model_name == 'random_forest':
        model = RandomForestClassifier(n_estimators=500, random_state=42)
    elif model_name == 'gradient_boosting':
        model = GradientBoostingClassifier(n_estimators=500, random_state=42)
    elif model_name == 'neural_network':
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        model = MLPClassifier(random_state=42, max_iter=5000)
    elif model_name == 'ridge':
        model = RidgeClassifier(random_state=42)
    elif model_name == 'ridge_LR':
        model = LogisticRegression(penalty='l2', random_state=42, solver='lbfgs', max_iter=10000)
    elif model_name == 'lasso_LR':
        model = LogisticRegression(penalty='l1', random_state=42, solver='saga', max_iter=10000)
    elif model_name == 'ElNet_LR':
        model = LogisticRegression(penalty='elasticnet', random_state=42, solver='saga', max_iter=10000, l1_ratio=0.5)
    elif model_name == 'svm':
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        model = SVC(kernel='linear', random_state=42)
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    
    # Fit the model
    model.fit(X_train, y_train)

    # Get hard prediction and probability (if available)
    y_pred = model.predict(X_test)[0]
    y_prob = model.predict_proba(X_test)[:, 1][0] if hasattr(model, 'predict_proba') else np.nan

    return y_pred, y_prob, model.coef_ if hasattr(model, 'coef_') else None

# helper function to calculate specificity
def calculate_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    return specificity

# helper function to calculate 95% CI
def bootstrap_auc_ci(y_true, y_score, n_bootstrap=1000, ci=0.95, random_state=42):
    rng = np.random.RandomState(random_state)
    aucs = []
    n = len(y_true)
    for i in range(n_bootstrap):
        indices = rng.choice(range(n), size=n, replace=True)
        if len(np.unique(y_true[indices])) < 2:
            # AUC is not defined if only one class present in sample
            continue
        aucs.append(roc_auc_score(y_true[indices], y_score[indices]))
    lower = np.percentile(aucs, 100 * (1 - ci) / 2)
    upper = np.percentile(aucs, 100 * (1 + ci) / 2)
    return lower, upper, np.array(aucs)

# helper function to calculated permutation p-value for AUC anlaysis
def permutation_auc_pvalue(y_true, y_score, n_permutations=1000, random_state=42):
    rng = np.random.RandomState(random_state)
    observed_auc = roc_auc_score(y_true, y_score)
    permuted_aucs = []
    for i in range(n_permutations):
        permuted_y = rng.permutation(y_true)
        permuted_aucs.append(roc_auc_score(permuted_y, y_score))
    p_value = np.mean(np.array(permuted_aucs) >= observed_auc)
    return p_value, np.array(permuted_aucs)

# Read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

# Read in methylation values
result_df = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv', index_col=0)
vmr_vals = result_df.iloc[:, 10:].T.astype(float)

# remove values from pheno where CR data is not available
pheno = pheno[pheno['CR_filt'] == 1]

# Merge phenotype data
merged_df = pd.merge(vmr_vals, pheno[['ChemoResponse']], left_index=True, right_index=True)

# convert chemoresponse data from string into int
merged_df['ChemoResponse'] = merged_df['ChemoResponse'].astype(int)

# Subtract 1 from each value in the ChemoResponse column to match requirements for statsmodels's logistic regression algorithm
merged_df['ChemoResponse'] = merged_df['ChemoResponse'] - 1

# Initialize summary variables
accuracy_list = []
precision_list = []
recall_list = []
f1_list = []
specificity_list = []
auc_list = []
excluded_count_list = []  # To keep track of how many samples are excluded in each run
times_list = []  # To track time taken

# Initialize a dataframe to store the coefficients for each LOOCV iteration
coef_df = pd.DataFrame(index=vmr_vals.columns)

true_outcomes = merged_df['ChemoResponse'].values

# initialize predicted probabilities matrix
predicted_probs = np.zeros(len(merged_df)) * np.nan

loo = LeaveOneOut()  # Initialize once since the splitting does not depend on data order

# Do you want to limit each loop to only CR-associated VMRs?
# This variable changes whether or not we limit the number of VMRs being used for our 
# average to only CR-assocaited or if we feed in all VMRs
filter_cr_predictor = False
pval_filter = 0.005

# Run the LOO-CV loop
start_time = time.time()  # Start timing

predicted_outcomes = np.zeros(len(merged_df)) * np.nan  # Initialize with NaNs

for train_index, test_index in loo.split(merged_df):
    train_df = merged_df.iloc[train_index].copy()
    test_df = merged_df.iloc[test_index].copy()

    # Univariate Logistic regression for each feature 
    summary_stats_fragments = []
    for feature in list(merged_df.columns[:-1]):
        try:
            X = sm.add_constant(train_df[[feature]])  # Add a constant term for the intercept
            model = sm.Logit(train_df['ChemoResponse'], X)
            result = model.fit(disp=0)  # disp=0 suppresses the output during the fitting process
            summary = {
                'Predictor': feature,
                'Coefficient': result.params[feature],
                'P-value': result.pvalues[feature],
                'Odds Ratio': np.exp(result.params[feature])
            }
            summary_stats_fragments.append(summary)
        except Exception as e:
            print(f"Could not fit a Logistic model for {feature}. Reason: {e}")

    logistic_stats_df = pd.DataFrame(summary_stats_fragments)
    
    if filter_cr_predictor:
        # Apply nominal p-value cutoff instead of FDR
        logistic_stats_df = logistic_stats_df[logistic_stats_df['P-value'] < pval_filter]
    
    logistic_stats_df.index = logistic_stats_df['Predictor']
    
    if not logistic_stats_df.empty:
        selected_features = logistic_stats_df['Predictor'].tolist()

        X_train = train_df[selected_features]
        y_train = train_df['ChemoResponse']
        X_test = test_df[selected_features]
        predicted_outcome, predicted_prob, feature_coefficients = fit_predict_model(model_name, X_train, y_train, X_test)
        predicted_outcomes[test_index] = predicted_outcome
        predicted_probs[test_index] = predicted_prob

        actual_outcome = test_df['ChemoResponse'].iloc[0]
        print(f'{len(logistic_stats_df)} features used for {model_name} prediction of {test_df.index[0]}: Predicted outcome: {predicted_outcome}, Actual Outcome: {actual_outcome}')
        
        if feature_coefficients is not None:
            coef_series = pd.Series(feature_coefficients[0], index=selected_features)  # Only get first array if multiclass
            coef_df[test_df.index[0]] = coef_series  # Add coefficients to the DataFrame column    
    else:
        predicted_outcomes[test_index] = np.nan  # Assign NaN if no significant features are found
        
# Exclude NaN values from predicted_outcomes before calculating statistics
valid_indices = ~np.isnan(predicted_outcomes)
valid_predicted_outcomes = predicted_outcomes[valid_indices]
valid_predicted_probs = predicted_probs[valid_indices]
valid_true_outcomes = true_outcomes[valid_indices]
excluded_count = np.sum(~valid_indices)
excluded_count_list.append(excluded_count)  # Keep track of excluded counts

if len(valid_predicted_outcomes) > 0:
    ...
    auc_roc = roc_auc_score(valid_true_outcomes, valid_predicted_probs)
else:
    auc_roc = np.nan

# Calculate accuracy and other statistics
if len(valid_predicted_outcomes) > 0:
    accuracy = accuracy_score(valid_true_outcomes, valid_predicted_outcomes)
    precision = precision_score(valid_true_outcomes, valid_predicted_outcomes)
    recall = recall_score(valid_true_outcomes, valid_predicted_outcomes)
    f1 = f1_score(valid_true_outcomes, valid_predicted_outcomes)
    specificity = calculate_specificity(valid_true_outcomes, valid_predicted_outcomes)
else:
    accuracy = precision = recall = f1 = specificity = np.nan

auc_list.append(auc_roc)
accuracy_list.append(accuracy)
precision_list.append(precision)
recall_list.append(recall)
f1_list.append(f1)
specificity_list.append(specificity)

# End timing
end_time = time.time()
elapsed_time = (end_time - start_time) / 60  # Convert time to minutes
times_list.append(elapsed_time)  # Save the time

# Logging progress with time
print(f"Complete in {elapsed_time:.2f} minutes. {excluded_count} samples removed due to insufficient univariately correlated VMRs.")

##### Run ROC Analysis #####
# Only plot if we have valid predictions
if len(valid_predicted_probs) > 0:
    # Compute ROC curve and ROC area
    fpr, tpr, thresholds = roc_curve(valid_true_outcomes, valid_predicted_probs)
    roc_auc = auc(fpr, tpr)
    lower_ci, upper_ci, aucs = bootstrap_auc_ci(valid_true_outcomes, valid_predicted_probs, n_bootstrap=1000)
    p_value, permuted_aucs = permutation_auc_pvalue(valid_true_outcomes, valid_predicted_probs, n_permutations=100000)
    print(f"AUC: {roc_auc:.3f}, 95% CI: [{lower_ci:.3f}, {upper_ci:.3f}], p-value: {p_value}")

    # Plot ROC curve
    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Lasso Regularized Logistic Regression')
    plt.legend(loc="lower right", fontsize=16)
    plt.grid(False)
    plt.tight_layout()
    plt.show()
else:
    print("No valid predictions available for ROC curve.")

In [ ]:
#### Supervised Chemoresponse Prediction (Signed Average) #####

# Dependencies
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

# Read in original methylation values
vmr_vals = pd.read_excel('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/sampleAvg_ONLY_vmr_using_dmrcate_beta_lam500.xlsx', index_col=0).T

# remove values from pheno where CR data is not available
pheno = pheno[pheno['CR_filt'] == 1]

# Merge phenotype data
merged_df = pd.merge(vmr_vals, pheno[['ChemoResponse']], left_index=True, right_index=True)

# convert chemoresponse data from string into int
merged_df['ChemoResponse'] = merged_df['ChemoResponse'].astype(int)

# Subtract 1 from each value in the ChemoResponse column to match requirements for statsmodels's logistic regression algorithm
merged_df['ChemoResponse'] = merged_df['ChemoResponse'] - 1

# HELPER FUNCTIONS
# function to calculate specificity
def calculate_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    return specificity

def bootstrap_auc_ci(y_true, y_score, n_bootstrap=2000, ci=0.95, random_state=42):
    rng = np.random.RandomState(random_state)
    aucs = []
    n = len(y_true)
    for i in range(n_bootstrap):
        indices = rng.choice(range(n), size=n, replace=True)
        if len(np.unique(y_true[indices])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[indices], y_score[indices]))
    lower = np.percentile(aucs, 100 * (1 - ci) / 2)
    upper = np.percentile(aucs, 100 * (1 + ci) / 2)
    return lower, upper, np.array(aucs)

def permutation_auc_pvalue(y_true, y_score, n_permutations=2000, random_state=42):
    rng = np.random.RandomState(random_state)
    observed_auc = roc_auc_score(y_true, y_score)
    permuted_aucs = []
    for i in range(n_permutations):
        permuted_y = rng.permutation(y_true)
        permuted_aucs.append(roc_auc_score(permuted_y, y_score))
    p_value = np.mean(np.array(permuted_aucs) >= observed_auc)
    return p_value, np.array(permuted_aucs)


# Initialize variables
accuracy_list = []
precision_list = []
recall_list = []
f1_list = []
specificity_list = []
excluded_count_list = []  # To keep track of how many samples are excluded in each run
times_list = []  # To track time taken

true_outcomes = merged_df['ChemoResponse'].values

loo = LeaveOneOut()  # Initialize once since the splitting does not depend on data order

# Run the LOO-CV loop
predicted_probabilities = np.zeros(len(merged_df)) * np.nan  # Store predicted probabilities for ROC

start_time = time.time()  # Start timing

predicted_outcomes = np.zeros(len(merged_df)) * np.nan  # Initialize with NaNs

for train_index, test_index in loo.split(merged_df):
    train_df = merged_df.iloc[train_index].copy()
    test_df = merged_df.iloc[test_index].copy()

    # Univariate Logistic regression for each feature 
    summary_stats_fragments = []
    for feature in list(vmr_vals.columns):
        try:
            X = sm.add_constant(train_df[[feature]])  # Add a constant term for the intercept
            univariate_model = sm.Logit(train_df['ChemoResponse'], X)
            result = univariate_model.fit(disp=0)  # disp=0 suppresses the output during the fitting process
            summary = {
                'Predictor': feature,
                'Coefficient': result.params[feature],
                'P-value': result.pvalues[feature],
                'Odds Ratio': np.exp(result.params[feature])
            }
            summary_stats_fragments.append(summary)
        except Exception as e:
            print(f"Could not fit a Logistic model for {feature}. Reason: {e}")

    logistic_stats_df = pd.DataFrame(summary_stats_fragments)

    # set fdr_cutoff
    fdr_cutoff = 1.0

    # run fdr correction and save only corrected p values
    _, p_corrected, _, _ = multipletests(logistic_stats_df['P-value'].values, alpha=fdr_cutoff, method='fdr_bh')

    # input corrected p values into stats dataframe
    logistic_stats_df['p_corrected'] = p_corrected

    # filter out values with corrected p-value lower than fdr_cutoff
    logistic_stats_df = logistic_stats_df[logistic_stats_df['P-value'] < fdr_cutoff]

    # set the index of the stats dataframe to be equal to the predictor value.  This helps in later steps
    logistic_stats_df.index = logistic_stats_df['Predictor']

    if not logistic_stats_df.empty:
        selected_features = logistic_stats_df['Predictor'].tolist()
        sign_mapping = logistic_stats_df['Coefficient'].apply(np.sign).to_dict()
        sa_train = train_df[selected_features].mul(sign_mapping, axis=1).mean(axis=1)
        sa_test = test_df[selected_features].mul(sign_mapping, axis=1).mean(axis=1)
        
        # Check for NaNs in sa_train and sa_test
        if not sa_train.isnull().any() and not sa_test.isnull().any():

            # add constant for training set
            sa_train_df = pd.DataFrame({'SignedAvg': sa_train, 'ChemoResponse': train_df['ChemoResponse']})
            sa_train_with_constant = sm.add_constant(sa_train_df['SignedAvg'])

            #print(f'sa_train_with_constant: {sa_train_with_constant}')

            # fit logistic model
            logistic_model = sm.Logit(sa_train_df['ChemoResponse'], sa_train_with_constant).fit(disp=0)
            
            # add constant for test set
            sa_test_df = pd.DataFrame({'SignedAvg': [sa_test.iloc[0]]}, index=[test_df.index[0]])
            sa_test_with_constant = sa_test_df.copy()
            sa_test_with_constant.insert(0, 'const', 1.0) # manually adding constant since

            # predict probability for test set
            predicted_probability = logistic_model.predict(sa_test_with_constant).values[0]
            predicted_probabilities[test_index] = predicted_probability  ### store for ROC curve

            # round probability to 0 or 1 - this converts probability to a binary outcome
            predicted_outcome = 1 if predicted_probability >= 0.5 else 0
            predicted_outcomes[test_index] = predicted_outcome
            
            actual_outcome = test_df['ChemoResponse'].iloc[0]

            print(f'{len(logistic_stats_df)} univariately significant VMRs found after FDR correction when removing {test_df.index[0]}. Predicted outcome of {test_df.index[0]}: {predicted_outcome}, Actual Outcome of {test_df.index[0]}: {actual_outcome}')
        else:
            print("NaNs found in sa_train or sa_test. Skipping this iteration.")
            predicted_outcomes[test_index] = np.nan  # Assign NaN if NaNs are found
    else:
        print(f'No significant VMRs found after correction when removing {test_df.index[0]}')
        predicted_outcomes[test_index] = np.nan  # Assign NaN if no significant features are found

# Exclude NaN values from predicted_outcomes before calculating statistics
valid_indices = ~np.isnan(predicted_outcomes)
valid_predicted_outcomes = predicted_outcomes[valid_indices]
valid_true_outcomes = true_outcomes[valid_indices]
excluded_count = np.sum(~valid_indices)
excluded_count_list.append(excluded_count)  # Keep track of excluded counts

valid_predicted_probs = predicted_probabilities[valid_indices]  # get probabilities where prediction is valid

# Calculate AUC
if len(np.unique(valid_true_outcomes)) > 1:  # Only calculate AUC if both classes are present
    roc_auc_val = roc_auc_score(valid_true_outcomes, valid_predicted_probs)
    # Bootstrap CI
    lower_ci, upper_ci, aucs = bootstrap_auc_ci(valid_true_outcomes, valid_predicted_probs, n_bootstrap=1000)
    # Permutation p-value
    p_value, permuted_aucs = permutation_auc_pvalue(valid_true_outcomes, valid_predicted_probs, n_permutations=100000)
    print(f"AUC: {roc_auc_val:.3f}, 95% CI: [{lower_ci:.3f}, {upper_ci:.3f}], p-value: {p_value}")

    # Plot ROC curve
    fpr, tpr, _ = roc_curve(valid_true_outcomes, valid_predicted_probs)
    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc_val:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('1-Specificity (FP)')
    plt.ylabel('Sensitivity (TP)')
    plt.title('ROC Curve for Signed Average CR Prediction')
    plt.legend(loc='lower right', fontsize=16)
    plt.grid(False)
    plt.tight_layout()
    plt.show()
else:
    roc_auc_val = np.nan
    print("AUC could not be computed because only one class is present in the true outcomes.")


# Calculate accuracy and other statistics
if len(valid_predicted_outcomes) > 0:
    accuracy = accuracy_score(valid_true_outcomes, valid_predicted_outcomes)
    precision = precision_score(valid_true_outcomes, valid_predicted_outcomes)
    recall = recall_score(valid_true_outcomes, valid_predicted_outcomes)
    f1 = f1_score(valid_true_outcomes, valid_predicted_outcomes)
    specificity = calculate_specificity(valid_true_outcomes, valid_predicted_outcomes)
else:
    accuracy = precision = recall = f1 = specificity = np.nan

accuracy_list.append(accuracy)
precision_list.append(precision)
recall_list.append(recall)
f1_list.append(f1)
specificity_list.append(specificity)

# End timing
end_time = time.time()
elapsed_time = (end_time - start_time) / 60  # Convert time to minutes
times_list.append(elapsed_time)  # Save the time

# Logging progress with time
print(f"Complete in {elapsed_time:.2f} minutes. {excluded_count} samples removed due to insufficient univariately correlated VMRs.")

# Save results to CSV
stats = pd.DataFrame({
    'Accuracy': accuracy_list,
    'Precision': precision_list,
    'Recall': recall_list,
    'Specificity': specificity_list,
    'F1 Score': f1_list,
    'AUC': roc_auc_val,
    'AUC_CI_Lower': lower_ci,
    'AUC_CI_Upper': upper_ci,
    'AUC_pvalue': p_value,
    'ExcludedSamples': excluded_count_list,
    'TimeInMinutes': times_list
})

# Optionally, save results
#results_df.to_csv('H:/...')

### Script Name: Comparison of Average Absolute (beta) Methylation Value by Risk Groups

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script analyzes the abosolute level of methylation by exploring the untransformed beta value in relation to global risk groups.

**Requires**: TARGET sample phenotype matrix, global VMR profile (Beta values), list of univariately significant survival-associated VMRs (and their respective assosiation scores)

In [ ]:
# Average Absolute Methylation by Risk Group
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns

# load data
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)
combined_beta = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_beta_vals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv')

combined_beta['vmr_name'] = (
    combined_beta['chr'].astype(str)
    + '-' + combined_beta['start'].astype(str)
    + '-' + combined_beta['end'].astype(str)
)

combined_beta = combined_beta.set_index('vmr_name')

# optionally filter to only 61 found in supervised analysis
uni_cox = pd.read_excel('H:/Projects/Methylation_Project/TARGET_Continued/univariate_cox_scores/mval_FDR_corrected_univariate_cox_scores.xlsx', index_col=0)
sig_vmrs = uni_cox[uni_cox['p_corrected'] < 0.05].index
combined_beta = combined_beta[combined_beta.index.isin(sig_vmrs)]

##### Average methylation per sample #####
umap_clust1_samples = pheno.index[pheno['risk_scores_FDR1_bin'] == 'Low']
umap_clust2_samples = pheno.index[pheno['risk_scores_FDR1_bin'] == 'High']

avg_meth_clust1 = combined_beta[umap_clust1_samples].mean(axis=0)
avg_meth_clust2 = combined_beta[umap_clust2_samples].mean(axis=0)


##### Build tidy dataframe for plotting #####
df_plot = pd.DataFrame({
    'Sample': list(avg_meth_clust1.index) + list(avg_meth_clust2.index),
    'Average_Methylation': list(avg_meth_clust1.values) + list(avg_meth_clust2.values),
    'Cluster': ['Low'] * len(avg_meth_clust1) + ['High'] * len(avg_meth_clust2)
})

##### Statistics #####
# Welch's t-test
t_stat, p_val = ttest_ind(avg_meth_clust1, avg_meth_clust2, equal_var=False)

# Cohen's d
def cohens_d(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    pooled_sd = np.sqrt(((nx - 1)*vx + (ny - 1)*vy) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_sd if pooled_sd > 0 else np.nan

d = cohens_d(avg_meth_clust1, avg_meth_clust2)

# Calculate Mean difference
mean_clust1 = avg_meth_clust1.mean()
mean_clust2 = avg_meth_clust2.mean()
mean_diff = mean_clust1 - mean_clust2

print("Average methylation per sample (Clust1 vs Clust2):")
print(f"  Clust1 mean ± SD: {mean_clust1:.4f} ± {avg_meth_clust1.std(ddof=1):.4f}")
print(f"  Clust2 mean ± SD: {mean_clust2:.4f} ± {avg_meth_clust2.std(ddof=1):.4f}")
print(f"\nMean difference (Clust1 - Clust2): {mean_diff:.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4e}")
print(f"Cohen's d: {d:.4f}")

##### Boxplot #####
plt.figure(figsize=(6, 5))

sns.boxplot(
    data=df_plot,
    x="Cluster",
    y="Average_Methylation",
    width=0.5,
    palette=["#af8dc3", "#7fbf7b"]
)

sns.stripplot(
    data=df_plot,
    x="Cluster",
    y="Average_Methylation",
    color="black",
    size=6,
    jitter=0.15,
    alpha=0.7
)

plt.title("Average Methylation (beta) by Risk Groups", fontsize=16)
plt.xlabel("")
plt.ylabel("Average Methylation (mean across VMRs)", fontsize=14)

plt.tight_layout()
plt.show()

### Script Name: Univariable Comparison of Average Absolute (beta) Methylation Value by Risk Groups

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script univariably analyzes the absolute methylation level (beta values) of VMRs in relation to survival-associated risk groups 

**Requires**: TARGET sample phenotype matrix and global VMR profile (Beta values)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# load data
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)
combined_beta = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_beta_vals/vmr_using_dmrcate_beta_lam500_sampleAvgs.csv')

combined_beta['vmr_name'] = (
    combined_beta['chr'].astype(str)
    + '-' + combined_beta['start'].astype(str)
    + '-' + combined_beta['end'].astype(str)
)

combined_beta = combined_beta.set_index('vmr_name')

results = []

for vmr in combined_beta.index:
    x = combined_beta.loc[vmr, umap_clust1_samples].astype(float)
    y = combined_beta.loc[vmr, umap_clust2_samples].astype(float)

    x = x.dropna()
    y = y.dropna()

    if len(x) < 3 or len(y) < 3:
        continue

    # Welch's t-test
    t_stat, p_val = ttest_ind(x, y, equal_var=False)

    # Cohen's d
    nx, ny = len(x), len(y)
    vx, vy = x.var(ddof=1), y.var(ddof=1)
    pooled_sd = np.sqrt(((nx - 1)*vx + (ny - 1)*vy) / (nx + ny - 2))
    d = (x.mean() - y.mean()) / pooled_sd if pooled_sd > 0 else np.nan

    results.append({
        'VMR': vmr,
        'mean_clust1': x.mean(),
        'mean_clust2': y.mean(),
        'mean_diff': x.mean() - y.mean(),
        't_stat': t_stat,
        'p_value': p_val,
        'cohens_d': d
    })

results_df = pd.DataFrame(results)
print(f"Tested {results_df.shape[0]} VMRs")

reject, qvals, _, _ = multipletests(
    results_df['p_value'],
    alpha=0.05,
    method='fdr_bh'
)

results_df['q_value'] = qvals
results_df['significant'] = reject
results_df['abs_mean_diff'] = results_df['mean_diff'].abs()

print("Min p-value:", results_df['p_value'].min())
print("Significant VMRs (p < 0.05):", len(results_df[results_df['p_value'] <= 0.05]))
print("Significant VMRs (FDR < 0.05):", results_df['significant'].sum())

results_df[results_df['q_value'] < 0.05].sort_values('abs_mean_diff')

### Script Name: Survival Analysis (Supervised)

**Authors**: Joshua Bowers

**Language**: Python

**Description**: Kaplan-Meier survival analsyis for unstratified and metastasis-stratified supervised clustering results

**Requires**: TARGET sample phenotype matrix

In [ ]:
##### Survival Analysis of Supervised Risk Groups ##### 

## User Input ##

survival_months = 'RFS_months'  # 'RFS_months' or 'Surv_months'
survival_cens   = 'RFS_cens'    # 'RFS_cens'   or 'Surv_cens'

###############

# Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test
import itertools

# read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Publication_Materials/Manuscript_Pheno.csv', index_col=0)

# Initialize the Kaplan-Meier fitter
kmf = KaplanMeierFitter()

# Plot setup
plt.figure(figsize=(10, 7))


for group in ['High', 'Low']:
    # Subset to the group
    group_data = pheno[pheno['risk_scores_FDR1_bin'] == group]
    
    # Fit the Kaplan-Meier estimator
    kmf.fit(durations=group_data[survival_months], event_observed=group_data[survival_cens], label=f'{group} Risk')
    
    # Plot the survival function
    kmf.plot_survival_function()

    # Get and print the median survival time
    median_survival = kmf.median_survival_time_
    print(f'Median survival time for {group} Risk: {median_survival} months')


#plt.title('')
plt.xlabel(' ', fontsize=16, fontweight='bold')
#plt.ylabel('Overall Survival Probability', fontsize=16, fontweight='bold')
plt.grid(False)
plt.yticks(fontsize=16, fontweight='bold')
plt.xticks(fontsize=16, fontweight='bold')  
ax = plt.gca()
ax.set_ylim(-0.05, 1.05)
plt.legend(fontsize=14)
#plt.legend([],[], frameon=False)
plt.show()

results = logrank_test(
    durations_A=pheno[pheno['risk_scores_FDR1_bin'] == 'High'][survival_months],
    durations_B=pheno[pheno['risk_scores_FDR1_bin'] == 'Low'][survival_months],
    event_observed_A=pheno[pheno['risk_scores_FDR1_bin'] == 'High'][survival_cens],
    event_observed_B=pheno[pheno['risk_scores_FDR1_bin'] == 'Low'][survival_cens]
)
print('Log-rank Test p-value:', results.p_value)

##### Binary HR (Cox model) between risk groups #####
# Make a small df for the Cox model
cox_df = pheno[[survival_months, survival_cens, 'risk_scores_FDR1_bin']].dropna().copy()

# High vs Low (Low = reference 0, High = 1)
cox_df['risk_bin'] = cox_df['risk_scores_FDR1_bin'].map({'Low': 0, 'High': 1})

# Fit Cox proportional hazards model
cph = CoxPHFitter()
cph.fit(cox_df[[survival_months, survival_cens, 'risk_bin']], duration_col=survival_months, event_col=survival_cens)

# Extract HR and CI for the binary cluster variable
summary = cph.summary.loc['risk_bin']
hr = summary['exp(coef)']
ci_low = summary['exp(coef) lower 95%']
ci_high = summary['exp(coef) upper 95%']
p_hr = summary['p']

hr_flipped = 1 / hr
ci_low_flipped = 1 / ci_high
ci_high_flipped = 1 / ci_low

print(f"HR (High Risk Group vs Low Risk Gorup): {hr:.2f} "
      f"(95% CI: {ci_low:.2f}-{ci_high:.2f}), p = {p_hr:.3g}")

print(f"HR (Low Risk Group vs High Risk Groups): {hr_flipped:.2f} "
      f"(95% CI: {ci_low_flipped:.2f}-{ci_high_flipped:.2f}), p = {p_hr:.3g}")


In [ ]:
##### Metastasis Stratified Survival Analysis of Supervised Risk Groups ##### 

## User Input ##

survival_months = 'RFS_months'  # 'RFS_months' or 'Surv_months'
survival_cens   = 'RFS_cens'    # 'RFS_cens'   or 'Surv_cens'

###############

# Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test
import itertools

# read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Publication_Materials/Manuscript_Pheno.csv', index_col=0)


# build anlaysis dataframe
surv_df = pheno[[survival_months, survival_cens, "Met_at_Dx", "risk_scores_FDR1_bin"]].copy()
surv_df = surv_df.dropna()

# Define grouping
surv_df["risk_group"] = surv_df["risk_scores_FDR1_bin"]
surv_df = surv_df.dropna(subset=["risk_group"])

# KM plotting
cluster_map = {"High": "High Risk", "Low": "Low Risk"}
color_map   = {"High Risk": "C0", "Low Risk": "C1"}

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(10, 7))

for group in ['High', 'Low']:
    for met_at_dx in sorted(surv_df['Met_at_Dx'].unique()):
        subset = surv_df[(surv_df['risk_group'] == group) & (surv_df['Met_at_Dx'] == met_at_dx)]
        if subset.empty:
            continue

        kmf.fit(durations=subset[survival_months], event_observed=subset[survival_cens])

        cluster_label = cluster_map[group]
        color = color_map[cluster_label]
        ls    = '--' if met_at_dx == 1 else '-'   # dashed for Met positive

        kmf.plot_survival_function(
            ax=ax,
            color=color,
            linestyle=ls,
            linewidth=2,
            ci_show=False,
            label=f'{cluster_label}, Met {"Positive" if met_at_dx == 1 else "Negative"}'
        )

        # Get and print the median survival time
        median_survival = kmf.median_survival_time_
        print(f'Median survival time for {group} risk, met {"positive" if met_at_dx == 1 else "negative"}: {median_survival} months')

# Aesthetics and labeling
ax.set_xlabel(' ', fontsize=16, fontweight='bold')
#ax.set_ylabel('OS Probability', fontsize=16, fontweight='bold')
ax.grid(False)
plt.yticks(fontsize=16, fontweight='bold')
plt.xticks(fontsize=16, fontweight='bold')
ax = plt.gca()
ax.set_ylim(-0.05, 1.05)
plt.legend(fontsize=14)

plt.show()

##### Multigroup log-rank across the 4 strata #####
# Note: This is not the same as stratified log-rank
surv_df['combined_group'] = (
    surv_df['risk_group'].map(cluster_map) +
    ', Met ' + surv_df['Met_at_Dx'].map({0: 'Negative', 1: 'Positive'})
)

multivariate_results = multivariate_logrank_test(
    event_durations=surv_df[survival_months],
    groups=surv_df['combined_group'],
    event_observed=surv_df[survival_cens]
)
print('Multivariate Log-rank Test p-value:')
print(multivariate_results.summary)

##### Pairwise log-rank tests across the 4 strata #####

# Make clean labels for the 4 combinations
surv_df['pair_group'] = (
    surv_df['risk_group'].map(cluster_map) +
    ', Met ' + surv_df['Met_at_Dx'].map({0: 'Negative', 1: 'Positive'})
)

groups = surv_df['pair_group'].unique()
groups = sorted(groups)  # nice ordering

print("\nPAIRWISE LOG-RANK TESTS:\n")

for g1, g2 in itertools.combinations(groups, 2):
    df1 = surv_df[surv_df['pair_group'] == g1]
    df2 = surv_df[surv_df['pair_group'] == g2]

    result = logrank_test(df1[survival_months], df2[survival_months], df1[survival_cens], df2[survival_cens])

    print(f"{g1}  vs  {g2}")
    print(f"p = {result.p_value:.4g}")
    print("----------------------------------")

##### Pairwise Cox models with HR flipped so higher-risk is on top #####

print("\nPAIRWISE COX HAZARD RATIOS (higher-risk vs lower-risk):\n")

cph = CoxPHFitter()

for g1, g2 in itertools.combinations(groups, 2):
    df_pair = surv_df[surv_df['pair_group'].isin([g1, g2])].copy()
    # Binary indicator: by default, g2 = 1, g1 = 0
    df_pair['group_bin'] = (df_pair['pair_group'] == g2).astype(int)

    # Fit Cox model
    cph.fit(
        df_pair[[survival_months, survival_cens, 'group_bin']],
        duration_col=survival_months,
        event_col=survival_cens
    )

    s = cph.summary.loc['group_bin']
    hr      = s['exp(coef)']
    ci_low  = s['exp(coef) lower 95%']
    ci_high = s['exp(coef) upper 95%']
    p       = s['p']

    # Flip if HR < 1 so the reported HR is always > 1
    # and corresponds to "higher hazard vs lower hazard"
    if hr >= 1:
        ref_group  = g1
        comp_group = g2
        hr_out, ci_low_out, ci_high_out = hr, ci_low, ci_high
    else:
        # invert HR and CI; swap group labels
        ref_group  = g2
        comp_group = g1
        hr_out     = 1.0 / hr
        ci_low_out = 1.0 / ci_high
        ci_high_out = 1.0 / ci_low

    print(f"{comp_group}  vs  {ref_group}")
    print(f"HR = {hr_out:.2f} (95% CI {ci_low_out:.2f}–{ci_high_out:.2f}), p = {p:.3g}")
    print("----------------------------------")


## Section 5: Validation Cohort Analysis

### Script Name: AECM Univariable Analysis

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script defines which VMRs of the AECM global profile individually associate with survival using logistic regression (Reminder: censoring data unavailable in AECM)

**Requires**: AECM sample phenotype matrix, list of AECM VMRs (VMRs overlapping at least one help-tagging methylation probe), list of TARGET CR-associated VMRs, list of TARGET RFS-associated VMRs 

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# 0 to 1 scale (beta-like values)
ht_vmrs = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/help_tagging/ht_beta-like_VMRs/ht_avg_accross_vmrs.csv', index_col=0)
ht_pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/help_tagging/HT_OS_pheno_data.csv', index_col=0)

# Load the subset VMR lists
cr_sig_associated_vmrs = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/cr_sig_associated_vmrs.csv')
rfs_sig_associated_vmrs = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/rfs_sig_associated_vmrs.csv')

# Extract the VMR IDs from the lists
cr_sig_vmrs = cr_sig_associated_vmrs['Predictor'].tolist()
rfs_sig_vmrs = rfs_sig_associated_vmrs['covariate'].tolist()

# subset ht_vmrs into cr and rfs associated section
ht_cr_sig_data = ht_vmrs[ht_vmrs.index.isin(cr_sig_vmrs)]
ht_rfs_sig_data = ht_vmrs[ht_vmrs.index.isin(rfs_sig_vmrs)]

p_threshold = 0.1

##### RFS VMRs #####
common = ht_rfs_sig_data.columns.intersection(ht_pheno.index)
X = ht_rfs_sig_data.loc[:, common]
y = pd.to_numeric(ht_pheno.loc[common, "5yr_EFS"], errors="coerce")

# z-score per VMR (row), ddof=1; drop zero-variance
sd = X.std(axis=1, ddof=1)
Xz = X.loc[sd > 0].sub(X.mean(axis=1), axis=0).div(sd, axis=0)

# analysis frame
df = Xz.T.join(y.rename("5yr_EFS")).dropna()
features = df.columns.drop("5yr_EFS")


# per-feature univariate logistic regression
summary_frags = []
for feature in features:
    Xi = sm.add_constant(df[[feature]])
    model = sm.Logit(df["5yr_EFS"].astype(float), Xi)
    res = model.fit(disp=0)
    summary = {
        'VMR': feature,
        'coef': res.params[feature],
        'p': res.pvalues[feature],
        'odds_ratio': np.exp(res.params[feature])
    }
    summary_frags.append(summary)

rfs_univariable = pd.DataFrame(summary_frags).sort_values("p")

# BH FDR
rfs_univariable["qval_bh"] = multipletests(rfs_univariable["p"].values, method="fdr_bh")[1]

# Defines how many of the 61 VMRs
print("Univariable RFS VRMs (Validation Cohort):", (rfs_univariable["p"] <= p_threshold).sum(), "/", len(rfs_univariable), "p <=", p_threshold)



##### CR VMRs #####
common = ht_cr_sig_data.columns.intersection(ht_pheno.index)
X = ht_cr_sig_data.loc[:, common]
y_raw = pd.to_numeric(ht_pheno.loc[common, "ChemoResponse"], errors="coerce")

# z-score per VMR (row), ddof=1; drop zero-variance
sd = X.std(axis=1, ddof=1)
Xz = X.loc[sd > 0].sub(X.mean(axis=1), axis=0).div(sd, axis=0)

# analysis frame
df = Xz.T.join(y_raw.rename("ChemoResponse")).dropna()
y = df["ChemoResponse"].astype(float)

### prefilter: drop constant and 1D separable features ###
def is_separable(x, y):
    x1 = x[y == 1]; x0 = x[y == 0]
    if x1.empty or x0.empty:
        return True
    return (x1.min() >= x0.max()) or (x0.min() >= x1.max())

features = [
    f for f in df.columns.drop("ChemoResponse")
    if df[f].nunique() > 1 and not is_separable(df[f], y)
]

### per-feature univariate logistic regression (plain MLE) ###
summary_frags = []
for feature in features:
    Xi = sm.add_constant(df[[feature]], has_constant="add")
    res = sm.Logit(y, Xi).fit(disp=0)
    summary_frags.append({
        "VMR": feature,
        "coef": float(res.params[feature]),
        "p": float(res.pvalues[feature]),
        "odds_ratio": float(np.exp(res.params[feature])),
    })

cr_univariable = pd.DataFrame(summary_frags).sort_values("p")

# BH FDR
cr_univariable["qval_bh"] = multipletests(cr_univariable["p"].values, method="fdr_bh")[1]

print("Univariable CR VMRs (Validation Cohort):", (cr_univariable["p"] <= p_threshold).sum(), "/", len(cr_univariable), "p <=", p_threshold)


### Script Name: AECM Clustermaps

**Authors**: Joshua Bowers

**Language**: Python

**Description**: Creates clustered heatmap of the global, CR-associated, and RFS-associated AECM VMR profile

**Requires**: AECM sample phenotype matrix, AECM VMRs data matrix, list of TARGET CR-associated VMRs, list of TARGET RFS-associated VMRs 

In [ ]:
# Dependencies
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
from matplotlib.patches import Patch

# 0 to 1 scale (beta-like values)
ht_vmrs = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/help_tagging/ht_beta-like_VMRs/ht_avg_accross_vmrs.csv', index_col=0)
ht_pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/help_tagging/HT_OS_pheno_data.csv', index_col=0)

# Load the subset VMR lists
cr_sig_associated_vmrs = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/cr_sig_associated_vmrs.csv')
rfs_sig_associated_vmrs = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/rfs_sig_associated_vmrs.csv')

# Extract the VMR IDs from the lists
cr_sig_vmrs = cr_sig_associated_vmrs['Predictor'].tolist()
rfs_sig_vmrs = rfs_sig_associated_vmrs['covariate'].tolist()

# subset ht_vmrs into cr and rfs associated section
ht_cr_sig_data = ht_vmrs[ht_vmrs.index.isin(cr_sig_vmrs)]
ht_rfs_sig_data = ht_vmrs[ht_vmrs.index.isin(rfs_sig_vmrs)]




##### RFS-Asscoiated VMRs Clustermap #####

# Select phenotype data of interest
phenotypes = ht_pheno[['ChemoResponse', '5yr_EFS', '5yr_OS', 'Metastasis']]

# Define color mappings
pheno_colors = {
    'ChemoResponse': {0: 'lightgrey', 1: 'blue'},
    '5yr_EFS': {0: 'lightgrey', 1: 'green'},
    '5yr_OS': {0: 'lightgrey', 1: 'red'},
    'Metastasis': {0: 'lightgrey', 1: 'purple'}
}

# Create a color DataFrame for phenotype ribbons
col_colors = pd.DataFrame(index=phenotypes.index)
for col in phenotypes.columns:
    col_colors[col] = phenotypes[col].map(pheno_colors[col])

# Define the legend patches for each phenotype
legend_elements = [
    Patch(facecolor='blue', edgecolor='k', label='ChemoResponse: Poor'),
    Patch(facecolor='lightgrey', edgecolor='k', label='ChemoResponse: Good'),
    Patch(facecolor='green', edgecolor='k', label='5yr_EFS: Relapse'),
    Patch(facecolor='lightgrey', edgecolor='k', label='5yr_EFS: No Relapse'),
    Patch(facecolor='red', edgecolor='k', label='5yr_OS: Dead'),
    Patch(facecolor='lightgrey', edgecolor='k', label='5yr_OS: Alive'),
    Patch(facecolor='purple', edgecolor='k', label='Metastasis: Yes'),
    Patch(facecolor='lightgrey', edgecolor='k', label='Metastasis: No')
]

# Generate the clustermap
g = sns.clustermap(
    ht_rfs_sig_data.apply(zscore, axis=1).dropna(),
    row_cluster=True,       # Cluster rows (VMRs)
    col_cluster=True,       # Cluster columns (Samples)
    cmap="vlag",            # Color map for the heatmap
    col_colors=col_colors,  # Add phenotype color ribbon
    figsize=(12, 8),        # Figure size
    metric="euclidean",     # Use Euclidean distance
    method="ward",      # Use complete linkage
    vmin=-2.5,
    vmax=2.5
)

# Remove row labels (VMR names)
g.ax_heatmap.set_yticklabels([])

# Adjust color bar position to the right
g.cax.set_position([0.85, 0.2, 0.03, 0.4])  # Moving the color bar to the right

# Add legend to the right side of the figure without overlap
g.figure.legend(
    handles=legend_elements,
    loc='center right',
    title="Phenotype Data",
    bbox_to_anchor=(1.1, 0.5),
    borderaxespad=0,
    frameon=False
)

g.figure.suptitle("RFS-Associated HELP-tagging VMRs", y=1.00)

# Show the plot
plt.show()




##### CR-Asscoiated VMRs - Clustermap #####

# Select phenotype data of interest
phenotypes = ht_pheno[['ChemoResponse', '5yr_EFS', '5yr_OS', 'Metastasis']]

# Define color mappings
pheno_colors = {
    'ChemoResponse': {0: 'lightgrey', 1: 'blue'},
    '5yr_EFS': {0: 'lightgrey', 1: 'green'},
    '5yr_OS': {0: 'lightgrey', 1: 'red'},
    'Metastasis': {0: 'lightgrey', 1: 'purple'}
}

# Create a color DataFrame for phenotype ribbons
col_colors = pd.DataFrame(index=phenotypes.index)
for col in phenotypes.columns:
    col_colors[col] = phenotypes[col].map(pheno_colors[col])

# Define the legend patches for each phenotype
legend_elements = [
    Patch(facecolor='blue', edgecolor='k', label='ChemoResponse: Poor'),
    Patch(facecolor='lightgrey', edgecolor='k', label='ChemoResponse: Good'),
    Patch(facecolor='green', edgecolor='k', label='5yr_EFS: Relapse'),
    Patch(facecolor='lightgrey', edgecolor='k', label='5yr_EFS: No Relapse'),
    Patch(facecolor='red', edgecolor='k', label='5yr_OS: Dead'),
    Patch(facecolor='lightgrey', edgecolor='k', label='5yr_OS: Alive'),
    Patch(facecolor='purple', edgecolor='k', label='Metastasis: Yes'),
    Patch(facecolor='lightgrey', edgecolor='k', label='Metastasis: No')
]

# Generate the clustermap
g = sns.clustermap(
    ht_cr_sig_data.apply(zscore, axis=1).dropna(),
    row_cluster=True,       # Cluster rows (VMRs)
    col_cluster=True,       # Cluster columns (Samples)
    cmap="vlag",            # Color map for the heatmap
    col_colors=col_colors,  # Add phenotype color ribbon
    figsize=(14, 10),        # Figure size
    metric="euclidean",     # Use Euclidean distance
    method="ward",
    vmin=-2.5,
    vmax=2.5
)

# Remove row labels (VMR names)
g.ax_heatmap.set_yticklabels([])

# Adjust color bar position to the right
g.cax.set_position([0.85, 0.2, 0.03, 0.4])  # Moving the color bar to the right

# Add legend to the right side of the figure without overlap
g.figure.legend(
    handles=legend_elements,
    loc='center right',
    title="Phenotype Data",
    bbox_to_anchor=(1.1, 0.5),
    borderaxespad=0,
    frameon=False
)

g.figure.suptitle("CR-Associated HELP-tagging VMRs", y=1.00)

# Show the plot
plt.show()




##### All VMRs - Clustermap #####

# Select phenotype data of interest
phenotypes = ht_pheno[['ChemoResponse', '5yr_EFS', '5yr_OS', 'Metastasis']]

# Define color mappings
pheno_colors = {
    'ChemoResponse': {0: 'lightgrey', 1: 'blue'},
    '5yr_EFS': {0: 'lightgrey', 1: 'green'},
    '5yr_OS': {0: 'lightgrey', 1: 'red'},
    'Metastasis': {0: 'lightgrey', 1: 'purple'}
}

# Create a color DataFrame for phenotype ribbons
col_colors = pd.DataFrame(index=phenotypes.index)
for col in phenotypes.columns:
    col_colors[col] = phenotypes[col].map(pheno_colors[col])

# Define the legend patches for each phenotype
legend_elements = [
    Patch(facecolor='blue', edgecolor='k', label='ChemoResponse: Poor'),
    Patch(facecolor='lightgrey', edgecolor='k', label='ChemoResponse: Good'),
    Patch(facecolor='green', edgecolor='k', label='5yr_EFS: Relapse'),
    Patch(facecolor='lightgrey', edgecolor='k', label='5yr_EFS: No Relapse'),
    Patch(facecolor='red', edgecolor='k', label='5yr_OS: Dead'),
    Patch(facecolor='lightgrey', edgecolor='k', label='5yr_OS: Alive'),
    Patch(facecolor='purple', edgecolor='k', label='Metastasis: Yes'),
    Patch(facecolor='lightgrey', edgecolor='k', label='Metastasis: No')
]

# Generate the clustermap
g = sns.clustermap(
    ht_vmrs.apply(zscore, axis=1).dropna(),
    row_cluster=True,       # Cluster rows (VMRs)
    col_cluster=True,       # Cluster columns (Samples)
    cmap="vlag",            # Color map for the heatmap
    col_colors=col_colors,  # Add phenotype color ribbon
    figsize=(14, 10),        # Figure size
    metric="euclidean",     # Use Euclidean distance
    method="ward",
    vmin=-2.5,
    vmax=2.5
)

# Remove row labels (VMR names)
g.ax_heatmap.set_yticklabels([])

# Adjust color bar position to the right
g.cax.set_position([0.85, 0.2, 0.03, 0.4])  # Moving the color bar to the right

# Add legend to the right side of the figure without overlap
g.figure.legend(
    handles=legend_elements,
    loc='center right',
    title="Phenotype Data",
    bbox_to_anchor=(1.1, 0.5),
    borderaxespad=0,
    frameon=False
)

g.figure.suptitle("All HELP-tagging VMRs", y=1.00)

# Show the plot
plt.show()

### Script Name: Supervised Survival Prediction (Validation Cohort)

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script defines a model on the TARGET methylation dataset and is applied to predict risk, expected long-term survival, and accompanying performance metrics for the AECM dataset

**Requires**: AECM sample phenotype matrix, AECM VMRs data matrix, TARGET sample phenotype matrix, and TARGET global VMR profile (M-values)

In [ ]:
## User Input ##

corrected = None  # 'CR' or 'Met_at_Dx', or None
P_stringency = 0.05
z_score = True

###############

# Dependencies
from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Helper Functions
def bootstrap_auc_ci(y_true, y_score, n_bootstrap=2000, ci=0.95, random_state=42, stratified=True):
    """
    Bootstrap percentile CI for ROC AUC.
    If stratified=True, resamples within each class to preserve class balance.
    Returns (lower, upper, auc_samples)
    """
    rng = np.random.RandomState(random_state)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    aucs = []
    n = len(y_true)

    if stratified:
        pos_idx = np.where(y_true == 1)[0]
        neg_idx = np.where(y_true == 0)[0]
        n_pos = len(pos_idx)
        n_neg = len(neg_idx)
        if n_pos == 0 or n_neg == 0:
            return np.nan, np.nan, np.array([])
        for _ in range(n_bootstrap):
            boot_pos = rng.choice(pos_idx, size=n_pos, replace=True)
            boot_neg = rng.choice(neg_idx, size=n_neg, replace=True)
            boot_idx = np.concatenate([boot_pos, boot_neg])
            aucs.append(roc_auc_score(y_true[boot_idx], y_score[boot_idx]))
    else:
        for _ in range(n_bootstrap):
            indices = rng.choice(np.arange(n), size=n, replace=True)
            if len(np.unique(y_true[indices])) < 2:
                continue
            aucs.append(roc_auc_score(y_true[indices], y_score[indices]))

    aucs = np.array(aucs)
    if aucs.size == 0:
        return np.nan, np.nan, aucs
    alpha = 1 - ci
    lower = np.percentile(aucs, 100 * (alpha / 2))
    upper = np.percentile(aucs, 100 * (1 - alpha / 2))
    return float(lower), float(upper), aucs


def permutation_auc_pvalue(y_true, y_score, n_permutations=10000, alternative="greater",
                           random_state=42, stratified=True):
    """
    Permutation p-value for AUC vs 0.5 (random). Keeps scores fixed, permutes labels.
    If stratified=True: permute *within* class counts (i.e., shuffle labels but keep prevalence the same).
    alternative: 'greater' (default), 'less', or 'two-sided'
    Returns (p_value, permuted_aucs)
    """
    rng = np.random.RandomState(random_state)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    observed_auc = roc_auc_score(y_true, y_score)
    n = len(y_true)
    permuted_aucs = []

    if stratified:
        # stratified label shuffling: reassign labels while preserving class counts
        n_pos = np.sum(y_true == 1)
        n_neg = n - n_pos
        for _ in range(n_permutations):
            # draw new labels with same counts
            perm_labels = np.zeros(n, dtype=int)
            perm_pos_idx = rng.choice(np.arange(n), size=n_pos, replace=False)
            perm_labels[perm_pos_idx] = 1
            permuted_aucs.append(roc_auc_score(perm_labels, y_score))
    else:
        for _ in range(n_permutations):
            perm_labels = rng.permutation(y_true)
            permuted_aucs.append(roc_auc_score(perm_labels, y_score))

    permuted_aucs = np.array(permuted_aucs)

    # Convert to a test statistic centered at 0.5
    obs_stat = observed_auc - 0.5
    perm_stats = permuted_aucs - 0.5

    if alternative == "greater":
        p = np.mean(perm_stats >= obs_stat)
    elif alternative == "less":
        p = np.mean(perm_stats <= obs_stat)
    elif alternative == "two-sided":
        p = np.mean(np.abs(perm_stats) >= abs(obs_stat))
    else:
        raise ValueError("alternative must be 'greater', 'less', or 'two-sided'")

    # add +1 correction to avoid zero p-value if desired:
    # p = (np.sum(condition) + 1) / (len(permuted_aucs) + 1)

    return float(p), permuted_aucs

##### Load and Prep Data #####
# read in phenotype data for samples
pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

# read in original methylation values
vmr_vals = pd.read_excel('H:/Projects/Methylation_Project/TARGET_Continued/VMRs/attatched_mvals/sampleAvg_ONLY_vmr_using_dmrcate_beta_lam500.xlsx', index_col=0)
vmr_vals = vmr_vals.T

merged_df = pd.merge(vmr_vals, pheno[['Surv_months', 'Surv_cens']], left_index=True, right_index=True)

# load in HELP-tagging VMRs
ht_vmrs = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/help_tagging/ht_mval-like_VMRs/ht_mval_vmrs.csv', index_col=0)
ht_vmrs = ht_vmrs.T

# load in HT pheno data
ht_pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Continued/help_tagging/HT_OS_pheno_data.csv', index_col=0)


##### Define Training Set #####
# Identify shared VMRs
shared_vmrs = list(set(vmr_vals.columns).intersection(set(ht_vmrs.columns)))
print(f"{len(shared_vmrs)} shared VMRs between training and validation set")

# Training set (subset to shared VMRs)
train_df = pd.merge(vmr_vals[shared_vmrs], pheno[['Surv_months', 'Surv_cens', 'ChemoResponse']], left_index=True, right_index=True)
if corrected == 'Met_at_Dx':
    train_df = train_df.drop('ChemoResponse', axis=1)

# Univariate Cox regression
cox_stats_df = pd.read_excel('H:/Projects/Methylation_Project/TARGET_Continued/univariate_cox_scores/mval_FDR_corrected_univariate_cox_scores.xlsx', index_col=0)

# remove VMRs not found in help-tagging dataset
cox_stats_df = cox_stats_df[cox_stats_df.index.isin(ht_vmrs.columns)]

# Select features + compute SignedAvg
significant_vmr_df = cox_stats_df[cox_stats_df['p_corrected'] < P_stringency]
selected_features = significant_vmr_df.index.tolist()
sign_mapping = significant_vmr_df['coef'].apply(np.sign).to_dict()

print(f"{len(selected_features)} significant features used")

if z_score:
    scaler = StandardScaler()
    train_scaled = pd.DataFrame(
        scaler.fit_transform(train_df[selected_features]),
        columns=selected_features,
        index=train_df.index
    )
    signed_train = train_scaled.copy()
else:
    signed_train = train_df[selected_features].copy()

# Apply sign mapping
for feat in selected_features:
    signed_train[feat] *= sign_mapping[feat]

# Compute signed average
train_df['SignedAvg'] = signed_train.mean(axis=1)

valid_idx = pheno.loc[train_df.index, 'Death_by_5y'] != ' '

# Apply the filter to both train_df and pheno
train_df = train_df.loc[valid_idx].copy()
train_df['Death_by_5y'] = pheno.loc[train_df.index, 'Death_by_5y'].astype(int)

X_train = train_df[['SignedAvg']].copy()
if corrected == 'CR':
    X_train['ChemoResponse'] = pheno.loc[X_train.index, 'ChemoResponse']
    X_train = pd.get_dummies(X_train, drop_first=True)
elif corrected == 'Met_at_Dx':
    X_train['Met_at_Dx'] = pheno.loc[X_train.index, 'Met_at_Dx'].astype("category")
    X_train = pd.get_dummies(X_train, drop_first=True)

y_train = train_df['Death_by_5y']

# Fit model
lr = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
lr.fit(X_train, y_train)

##### Define Validation Set #####
val_df = pd.merge(ht_vmrs[selected_features], ht_pheno[['ChemoResponse', '5yr_OS']], left_index=True, right_index=True)

if z_score:
    val_scaled = pd.DataFrame(
        scaler.transform(val_df[selected_features]),
        columns=selected_features,
        index=val_df.index
    )
    signed_test = val_scaled.copy()
else:
    signed_test = val_df[selected_features].copy()

# apply sign mapping
for feat in selected_features:
    signed_test[feat] *= sign_mapping[feat]

# Compute signed average
val_df['SignedAvg'] = signed_test.mean(axis=1)

X_val = val_df[['SignedAvg']].copy()
if corrected == 'CR':
    X_val['ChemoResponse'] = ht_pheno.loc[X_val.index, 'ChemoResponse']
    X_val = pd.get_dummies(X_val, drop_first=True)
elif corrected == 'Met_at_Dx':
    X_val['Met_at_Dx'] = ht_pheno.loc[X_val.index, 'Metastasis'].astype("category")
    X_val = pd.get_dummies(X_val, drop_first=True)

# Match validation columns to training
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
y_val = val_df['5yr_OS'].astype(int)

# Predict
val_probs = lr.predict_proba(X_val)[:, 1]
val_preds = lr.predict(X_val)

# Evaluation
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
auc_val = roc_auc_score(y_val, val_probs)
print("Validation AUC:", auc_val)
print(pd.crosstab(y_val, val_preds))

##### Calculate AUC CI and P-value #####
# Bootstrap percentile CI (stratified)
ci_low, ci_high, boot_aucs = bootstrap_auc_ci(
    y_true=y_val.values,
    y_score=val_probs,
    n_bootstrap=2000,
    ci=0.95,
    random_state=42,
    stratified=True
)

p_perm, perm_aucs = permutation_auc_pvalue(
    y_true=y_val.values,
    y_score=val_probs,
    n_permutations=10000,
    alternative="two-sided",
    random_state=42,
    stratified=True
)

print(f"AUC (validation) = {auc_val:.3f} "
      f"(95% bootstrap CI: {ci_low:.3f}-{ci_high:.3f}); "
      f"Permutation p-value (AUC>0.5): p = {p_perm:.3g}")

# Plot ROC with CI in the legend
fpr, tpr, _ = roc_curve(y_val, val_probs)
plt.plot(fpr, tpr, label=f"AUC = {auc_val:.2f} (95% CI {ci_low:.2f}-{ci_high:.2f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("1 - Specificity (FP)")
plt.ylabel("Sensitivity (TP)")
# plt.title("5-Year OS")
plt.legend(loc="lower right", fontsize=12)
plt.grid(False)
plt.show()


## Section 6: Epigenetic Clock Analysis

### Script Name: TARGET Horvath Age Prediction

**Authors**: Nikhil Ramavenkat

**Language**: Python

**Description**: This script loads TARGET clinical and methylation data, reconstructs the Horvath 2013 epigenetic clock from published supplementary files, converts TARGET methylation M-values to beta values, aligns CpGs to the Horvath feature set, imputes missing CpGs using published reference values, and generates sample-level epigenetic age predictions.

**Requires**: TARGET clinical phenotype matrix, TARGET filtered methylation M-value matrix, and the ability to download Horvath supplementary files

In [ ]:
import pandas as pd

# Load TARGET phenotype/clinical annotations
clinical_data = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv')

# Load filtered TARGET methylation matrix in M-value format
# Rows = CpGs, columns = samples
mval_matrix = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/reproduced_TARGET_funnorm_Mval_filt.csv', index_col=0)

import os
import torch
import pyaging as pya

# Initialize the Horvath 2013 methylation age model
model = pya.models.Horvath2013()

# Manually define model metadata for recordkeeping
model.metadata = {
    "clock_name": "horvath2013",
    "data_type": "methylation",
    "species": "Homo sapiens",
    "year": 2013,
    "approved_by_author": "⌛",
    "citation": 'Horvath, Steve. "DNA methylation age of human tissues and cell types." Genome Biology 14.10 (2013): 1-20.',
    "doi": "https://doi.org/10.1186/gb-2013-14-10-r115",
    "research_only": None,
    "notes": None
}

# Download Horvath 2013 clock coefficients
coefficients_url = "https://static-content.springer.com/esm/art%3A10.1186%2Fgb-2013-14-10-r115/MediaObjects/13059_2013_3156_MOESM3_ESM.csv"
os.system(f"curl -o coefficients.csv {coefficients_url}")

# Download Horvath 2013 reference CpG values used for missing-feature imputation
reference_values_url = "https://static-content.springer.com/esm/art%3A10.1186%2Fgb-2013-14-10-r115/MediaObjects/13059_2013_3156_MOESM22_ESM.csv"
os.system(f"curl -o reference_feature_values.csv {reference_values_url}")

# Load published Horvath coefficients
coef_df = pd.read_csv("coefficients.csv", skiprows=2)
coef_df["feature"] = coef_df["CpGmarker"]
coef_df["coefficient"] = coef_df["CoefficientTraining"]

# Extract CpG features used by the clock
model.features = coef_df["feature"][1:].tolist()

# Build model weight matrix and intercept from the published coefficients
weights = torch.tensor(coef_df["coefficient"][1:].tolist()).unsqueeze(0)
intercept = torch.tensor([coef_df["coefficient"][0]])

# Recreate the linear prediction layer used by the Horvath clock
base_model = pya.models.LinearModel(input_dim=len(model.features))
base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()
model.base_model = base_model

# Load reference beta values for CpGs expected by the model
reference_feature_values_df = pd.read_csv("reference_feature_values.csv", index_col=0)
reference_feature_values_df = reference_feature_values_df.loc[model.features]
model.reference_values = reference_feature_values_df["goldstandard2"].tolist()

# Define postprocessing behavior used to convert model output to predicted age
model.preprocess_name = None
model.preprocess_dependencies = None
model.postprocess_name = "anti_log_linear"
model.postprocess_dependencies = None

# Print model summary for validation
pya.utils.print_model_details(model)

import os
import torch
import pandas as pd
import anndata
import pyaging as pya

# Ensure methylation matrix is numeric before conversion
mval_matrix_numeric = mval_matrix.apply(pd.to_numeric, errors="coerce")

# Convert M-values to beta values for Horvath clock input
beta_values = 2 ** mval_matrix_numeric.values / (2 ** mval_matrix_numeric.values + 1)
beta_matrix = pd.DataFrame(
    beta_values,
    index=mval_matrix.index,
    columns=mval_matrix.columns
)

# Build sample-by-feature input matrix expected by the Horvath model
# Missing CpGs are filled using Horvath reference values
data = pd.DataFrame(index=beta_matrix.columns, columns=model.features)

for cpg in model.features:
    if cpg in beta_matrix.index:
        data[cpg] = beta_matrix.loc[cpg]
    else:
        ref_idx = model.features.index(cpg)
        data[cpg] = model.reference_values[ref_idx]

# Convert to AnnData format for compatibility with pyaging utilities
adata = anndata.AnnData(X=data)

# Report overlap between TARGET CpGs and Horvath clock CpGs
print(f"Data shape: {data.shape}")
print(f"Number of samples: {len(data.index)}")
print(f"Number of CpGs used: {len(model.features)}")
print(f"Number of overlapping CpGs: {sum([cpg in beta_matrix.index for cpg in model.features])}")

# Save model object for reuse
os.makedirs("model_weights", exist_ok=True)
torch.save(model, f"model_weights/{model.metadata['clock_name']}.pt")

# Set model to evaluation mode prior to prediction
model.eval()
model.to(torch.float)

# Predict epigenetic age
try:
    input_tensor = torch.tensor(data.values, dtype=torch.float)
    with torch.no_grad():
        predictions = model(input_tensor)

    print("\nDirect predictions successful!")
    print(predictions)

except Exception as e:
    print(f"Direct prediction failed with error: {e}")

    # Fallback to pyaging wrapper if direct inference fails
    try:
        ages = pya.pred.predict_age(adata, ["Horvath2013"])
        print("\nPredict_age successful!")
        print(ages.obs["Horvath2013"])
    except Exception as e:
        print(f"Predict_age failed with error: {e}")

# Store predictions in a sample-level DataFrame
predicted_ages = pd.DataFrame(
    predictions.numpy(),
    index=data.index,
    columns=["Predicted_Age"]
)

# Summarize predicted age distribution across samples
print("\nSummary Statistics:")
print("------------------")
print(f"Mean Age: {predicted_ages['Predicted_Age'].mean():.2f}")
print(f"Median Age: {predicted_ages['Predicted_Age'].median():.2f}")
print(f"Min Age: {predicted_ages['Predicted_Age'].min():.2f}")
print(f"Max Age: {predicted_ages['Predicted_Age'].max():.2f}")
print(f"Standard Deviation: {predicted_ages['Predicted_Age'].std():.2f}")

# Save predictions in a simple sample-level table
results_df = pd.DataFrame({
    "Sample_ID": predicted_ages.index,
    "Predicted_Age": predictions.numpy().flatten()
})

### Script Name: TARGET Age Acceleration Distribution

**Authors**: Nikhil Ramavenkat

**Language**: Python

**Description**: This script merges predicted Horvath ages with TARGET phenotype data, calculates age acceleration as predicted age minus chronological age, and visualizes the sample distribution of age acceleration.

**Requires**: TARGET clinical phenotype matrix, sample-level predicted age table

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Extract chronological age from TARGET clinical annotations
chronological_age = clinical_data.set_index("Sample_ID")["AgeDx"]

# Extract predicted epigenetic age from Horvath output
predicted_age = results_df.set_index("Sample_ID")["Predicted_Age"]

# Restrict analysis to samples present in both tables
common_samples = chronological_age.index.intersection(predicted_age.index)

# Build combined analysis table
analysis_df = pd.DataFrame({
    "Sample_ID": common_samples,
    "Chronological_Age": chronological_age.loc[common_samples].values,
    "Predicted_Age": predicted_age.loc[common_samples].values
})

# Define age acceleration as predicted epigenetic age minus chronological age
analysis_df["Age_Acceleration"] = (
    analysis_df["Predicted_Age"] - analysis_df["Chronological_Age"]
)

# Plot distribution of age acceleration across TARGET samples
plt.figure(figsize=(8, 6))
sns.histplot(
    data=analysis_df,
    x="Age_Acceleration",
    bins=20,
    edgecolor="black"
)

# Reference line at zero age acceleration
plt.axvline(x=0, color="red", linestyle="--", linewidth=1.5)

plt.title("Distribution of Age Acceleration", fontsize=16, fontweight="bold")
plt.xlabel("Age Acceleration (Predicted - Chronological)", fontsize=14)
plt.ylabel("Frequency", fontsize=14)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.tight_layout()
plt.show()

### Script Name: TARGET Epigenetic Age Association Analysis

**Authors**: Nikhil Ramavenkat

**Language**: Python

**Description**: This script evaluates whether predicted epigenetic age and age acceleration associate with chemotherapy response and relapse-free survival in TARGET. Chemotherapy response is tested using two-group comparisons, and relapse-free survival is evaluated using Kaplan–Meier analysis with median-based stratification.

**Requires**: TARGET clinical phenotype matrix, sample-level predicted age table

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, mannwhitneyu, shapiro

predicted_ages = results_df.copy()

# Merge phenotype and predicted epigenetic age
df = clinical_data.merge(predicted_ages, on="Sample_ID")

# Recode chemotherapy response and remove missing values
df["ChemoResponse"] = pd.to_numeric(df["ChemoResponse"], errors="coerce")
df = df.dropna(subset=["ChemoResponse"])
df["ChemoResponse_Binary"] = df["ChemoResponse"].replace({1: 0, 2: 1})

# Compute age acceleration
df["Age_Diff"] = df["Predicted_Age"] - df["AgeDx"]

def run_stats_and_plot(data, yvar, title, palette):
    group0 = data[data["ChemoResponse_Binary"] == 0][yvar]
    group1 = data[data["ChemoResponse_Binary"] == 1][yvar]

    # Assess normality within each response group to choose a statistical test
    normal0 = shapiro(group0)[1] > 0.05
    normal1 = shapiro(group1)[1] > 0.05

    if normal0 and normal1:
        stat, p = ttest_ind(group0, group1)
        test_used = "t-test"
    else:
        stat, p = mannwhitneyu(group0, group1, alternative="two-sided")
        test_used = "Mann-Whitney"

    # Assign significance label for plotting
    if p < 0.001:
        sig = "***"
    elif p < 0.01:
        sig = "**"
    elif p < 0.05:
        sig = "*"
    else:
        sig = "ns"

    # Plot group comparison
    plt.figure(figsize=(8, 6))
    ax = sns.boxplot(x="ChemoResponse_Binary", y=yvar, data=data, palette=palette)
    sns.stripplot(
        x="ChemoResponse_Binary", y=yvar, data=data,
        color="black", alpha=0.5, jitter=0.2
    )

    # Add significance bracket above groups
    y_max = data[yvar].max()
    y_min = data[yvar].min()
    rng = max(1e-6, y_max - y_min)

    h = 0.05 * rng
    y_bracket = y_max + h

    ax.plot([0, 0, 1, 1],
            [y_bracket, y_bracket + h, y_bracket + h, y_bracket],
            lw=1.5, c="k")

    ax.text(
        0.5, y_bracket + h * 1.1, sig,
        ha="center", va="bottom",
        fontsize=16, fontweight="bold",
        clip_on=False
    )

    ax.set_ylim(y_min - 0.05 * rng, y_bracket + h * 3)

    ax.set_xlabel(
        "Chemotherapy Response (0 = Suboptimal, 1 = Optimal)",
        fontsize=16, fontweight="bold"
    )
    ax.set_ylabel(yvar.replace("_", " "), fontsize=16, fontweight="bold")
    ax.set_title(f"{title}\n{test_used} p = {p:.4f}", fontsize=18, fontweight="bold")

    ax.tick_params(axis="both", labelsize=14)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")

    ax.grid(False)

    plt.tight_layout()
    plt.show()

    print(f"{title} — {test_used} p-value: {p:.4e} ({sig})")

# Compare predicted age and age acceleration by chemotherapy response group
run_stats_and_plot(df, "Predicted_Age", "Predicted Age by Chemotherapy Response", "Set2")
run_stats_and_plot(df, "Age_Diff", "Age Acceleration by Chemotherapy Response", "Set1")

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# Merge phenotype and predicted age data
df = clinical_data.merge(predicted_ages, on="Sample_ID")

# Compute age acceleration
df["Age_Acceleration"] = df["Predicted_Age"] - df["AgeDx"]

# Remove samples missing required covariates or survival information
df = df.dropna(subset=["Predicted_Age", "AgeDx", "Age_Acceleration", "RFS_months", "RFS_cens"])

# -----------------------------
# KM analysis: Predicted age
# -----------------------------
median_epigenetic_age = df["Predicted_Age"].median()
df["EpiAge_Group"] = df["Predicted_Age"].apply(
    lambda x: "High Epigenetic Age" if x > median_epigenetic_age else "Low Epigenetic Age"
)

df_high_epi = df[df["EpiAge_Group"] == "High Epigenetic Age"]
df_low_epi = df[df["EpiAge_Group"] == "Low Epigenetic Age"]

kmf_high_epi = KaplanMeierFitter()
kmf_low_epi = KaplanMeierFitter()

kmf_high_epi.fit(
    df_high_epi["RFS_months"],
    event_observed=df_high_epi["RFS_cens"],
    label="High Epigenetic Age"
)
kmf_low_epi.fit(
    df_low_epi["RFS_months"],
    event_observed=df_low_epi["RFS_cens"],
    label="Low Epigenetic Age"
)

plt.figure(figsize=(10, 6))
ax = kmf_high_epi.plot_survival_function(color="#ff7f0e", ci_show=False, linewidth=2)
kmf_low_epi.plot_survival_function(ax=ax, color="#1f77b4", ci_show=False, linewidth=2)

plt.title("Kaplan-Meier: Epigenetic Age Groups", fontsize=18, fontweight="bold")
plt.xlabel("Time (Months)", fontsize=16, fontweight="bold")
plt.ylabel("Survival Probability", fontsize=16, fontweight="bold")

plt.legend(fontsize=15, frameon=False)
ax.tick_params(axis="both", labelsize=14)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.grid(False)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

logrank_epi = logrank_test(
    df_high_epi["RFS_months"], df_low_epi["RFS_months"],
    event_observed_A=df_high_epi["RFS_cens"],
    event_observed_B=df_low_epi["RFS_cens"]
)
print(f"Log-Rank (Epigenetic Age): chi2 = {logrank_epi.test_statistic:.2f}, p = {logrank_epi.p_value:.4f}")

# -----------------------------
# KM analysis: Age acceleration
# -----------------------------
median_accel = df["Age_Acceleration"].median()
df["Accel_Group"] = df["Age_Acceleration"].apply(
    lambda x: "High Acceleration" if x > median_accel else "Low Acceleration"
)

df_high_accel = df[df["Accel_Group"] == "High Acceleration"]
df_low_accel = df[df["Accel_Group"] == "Low Acceleration"]

kmf_high_accel = KaplanMeierFitter()
kmf_low_accel = KaplanMeierFitter()

kmf_high_accel.fit(
    df_high_accel["RFS_months"],
    event_observed=df_high_accel["RFS_cens"],
    label="High Δ Age"
)
kmf_low_accel.fit(
    df_low_accel["RFS_months"],
    event_observed=df_low_accel["RFS_cens"],
    label="Low Δ Age"
)

plt.figure(figsize=(10, 6))
ax = kmf_high_accel.plot_survival_function(color="#ff7f0e", ci_show=False, linewidth=2)
kmf_low_accel.plot_survival_function(ax=ax, color="#1f77b4", ci_show=False, linewidth=2)

plt.title("Kaplan-Meier: Δ Age-Based Risk Groups", fontsize=18, fontweight="bold")
plt.xlabel("Time (Months)", fontsize=16, fontweight="bold")
plt.ylabel("Survival Probability", fontsize=16, fontweight="bold")

plt.legend(fontsize=15, frameon=False)
ax.tick_params(axis="both", labelsize=14)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.grid(False)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

logrank_accel = logrank_test(
    df_high_accel["RFS_months"], df_low_accel["RFS_months"],
    event_observed_A=df_high_accel["RFS_cens"],
    event_observed_B=df_low_accel["RFS_cens"]
)
print(f"Log-Rank (Age Acceleration): chi2 = {logrank_accel.test_statistic:.2f}, p = {logrank_accel.p_value:.4f}")

## Section 7: Multi-Platform Clustering and Comparative Anaysis

### Script Name: RNA-seq (Gene Expression) Clustering

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script produces a clustered heatmap of the TARGET RNA-seq-based gene expression dataset with 90% MAD (median absolute deviation - reduces effect of single-sample outliers that seem to be common in this dataset) filtering.

**Requires**: TARGET RNA-seq sample phenotype matrix and TARGET RNA-seq data matrix

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Define MAD function - more resistant to outliers
def mad(arr):
    return np.median(np.abs(arr - np.median(arr)))

# load phenotype data for RNA-seq
expr_pheno = pd.read_excel('H:/Projects/Integrative_Clustering/RNAseq/Global.xlsx', sheet_name='Experiment descriptors', index_col=0)

# load in RNA-seq gene expression data
expr_data = pd.read_csv('H:/Projects/Integrative_Clustering/RNAseq/rlog_expr_data.csv', index_col=0)

# Drop three potential outliers
#expr_data = expr_data.drop(columns=['0A4HLD','PATUXZ','PAPNVD'])

# transpose so that rows = samples, and columns = genes
expr_data_t = expr_data.T

# Standardize the data (important for clustering) - This is actually the same thing as zscoring
scaler = StandardScaler()
scaled_data = scaler.fit_transform(expr_data_t) # normalizes across gene per smaple

scaled_data = pd.DataFrame(scaled_data, index=expr_data_t.index, columns=expr_data_t.columns) # convert back to a data frame

# remove values with zero variance
scaled_data = scaled_data.loc[:, scaled_data.var(axis=0) > 0]

expr_pheno = expr_pheno[expr_pheno.index.isin(expr_data_t.index)]

# Median Absolute Deviation (MAD) filtering:

# Compute MAD for each gene (row-wise)
mad_values = np.apply_along_axis(mad, 1, expr_data)

# Select top 10% most variable genes
perc_thresh = 0.90
numeric_thresh = np.quantile(mad_values, perc_thresh)
mad_filt_expr_data = expr_data[mad_values >= numeric_thresh]

filt_expr_data_t = mad_filt_expr_data.T

# Standardize the data (important for clustering) - This is actually the same thing as zscoring
filt_scaler = StandardScaler()
filt_scaled_data = filt_scaler.fit_transform(filt_expr_data_t) # normalizes across gene per smaple

filt_scaled_data = pd.DataFrame(filt_scaled_data, index=filt_expr_data_t.index, columns=filt_expr_data_t.columns) # transform back into a dataframe

# Define color mapping for "Chemoresponse" (binary: 1 = suboptimal, 2 = optimal)
necrosis_palette = {1: "#d8b365", 2: "#5ab4ac"}  # Adjust colors if needed

# Define color mapping for "Recurrence by 5yrs" (binary: 0 = No Recurrence, 1 = Recurrence)
recurrence_palette = {0: "#2166ac", 1: "#b2182b"}  # Adjust colors if needed

# coerce to numeric
necrosis_vals   = pd.to_numeric(expr_pheno["Necrosis Category"], errors="coerce")
recurrence_vals = pd.to_numeric(expr_pheno["Recurrence by 5 yrs"], errors="coerce")

col_colors = pd.DataFrame({
    "Chemoresponse": necrosis_vals.map(necrosis_palette).fillna('#e0e0e0'),
    "Recurrence by 5yr": recurrence_vals.map(recurrence_palette).fillna('#e0e0e0'),
}, index=expr_pheno.index)


# Hierarchical clustering heatmap with phenotype annotations
g = sns.clustermap(
    filt_scaled_data.T, 
    method="ward", 
    cmap="coolwarm", 
    row_cluster=True, 
    col_cluster=True,  # Cluster only samples
    col_colors=col_colors,  # Add phenotype bars
    figsize=(12, 8),
    vmin=-3,  # Set the lower bound of the color scale
    vmax=3,   # Set the upper bound of the color scale
    yticklabels=False,  # Remove row labels
    xticklabels=False,
    cbar_pos=(0.9, 0.2, 0.03, 0.4),  # Move colorbar to the right
    dendrogram_ratio=(0.1, 0.1)
)

cbar = g.ax_heatmap.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)  # bigger tick text
cbar.set_label("Z-score", fontsize=16, fontweight='bold')  # set your label (not "ARI")
cbar.outline.set_linewidth(1.0)  # slightly thicker border

plt.suptitle("RNA-Seq Hierarchical Clustering Heatmap - 90% MAD Filtered", 
             x=0.5, y=1.05, fontsize=14)
plt.show()

### Script Name: miRNA Expression Clustering

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script produces a clustered heatmap of the TARGET miRNA expression dataset with 50% variance filtering.

**Requires**: TARGET miRNA expression sample phenotype matrix and TARGET miRNA expression data matrix

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# load phenotype data for miRNA
mirna_pheno = pd.read_excel('H:/Projects/Integrative_Clustering/miRNA/2negdCT.xlsx', sheet_name='Experiment descriptors', index_col=1)

# load in mirna expression data
mirna_data = pd.read_excel('H:/Projects/Integrative_Clustering/miRNA/2negdCT.xlsx', sheet_name='Filtered log intensity', index_col=0)

mirna_data = mirna_data.drop(columns=['P-Value', 'Rank', 'Variance', 'Num 1.5-Fold', 'Filter', 'Missing'])

# remove miRNA with one na value
mirna_data = mirna_data.dropna()

# transpose so that rows = samples, and columns = genes
mirna_data_t = mirna_data.T

# Standardize the data (important for clustering) - This is actually the same thing as zscoring
mirna_scaler = StandardScaler()
mirna_scaled_data = mirna_scaler.fit_transform(mirna_data_t) # normalizes across gene per smaple

mirna_scaled_data = pd.DataFrame(mirna_scaled_data, index=mirna_data_t.index, columns=mirna_data_t.columns)

# calculate gene variance
mirna_variances = mirna_data.var(axis=1)

# Select top 50% most variable genes
perc_thresh = 0.50
numeric_thresh = np.quantile(mirna_variances, perc_thresh)
filt_mirna_data = mirna_data[mirna_variances >= numeric_thresh]

filt_mirna_data_t = filt_mirna_data.T

# Standardize the data (important for clustering) - This is actually the same thing as zscoring
filt_mirna_scaler = StandardScaler()
filt_mirna_scaled_data = filt_mirna_scaler.fit_transform(filt_mirna_data_t) # normalizes across gene per smaple

# transform back into data frame with indicies and columns - easier to work with
filt_mirna_scaled_data = pd.DataFrame(filt_mirna_scaled_data, index=filt_mirna_data_t.index, columns=filt_mirna_data_t.columns)

# Define color mapping for "Chemoresponse" (binary: 1 = suboptimal, 2 = optimal)
necrosis_palette = {1: "#d8b365", 2: "#5ab4ac"}  # Adjust colors if needed

# Define color mapping for "Recurrence by 5yrs" (binary: 0 = No Recurrence, 1 = Recurrence)
recurrence_palette = {0: "#2166ac", 1: "#b2182b"}  # Adjust colors if needed

# (Optional) coerce to numeric in case your columns are strings like "1", "2", etc.
necrosis_vals   = pd.to_numeric(mirna_pheno["Necrosis Category"], errors="coerce")
recurrence_vals = pd.to_numeric(mirna_pheno["Recurrence by 5 yrs"], errors="coerce")

col_colors = pd.DataFrame({
    "Chemoresponse": necrosis_vals.map(necrosis_palette).fillna('#e0e0e0'),
    "Recurrence by 5yr": recurrence_vals.map(recurrence_palette).fillna('#e0e0e0'),
}, index=mirna_pheno.index)

# Hierarchical clustering heatmap with phenotype annotations
sns.clustermap(
    filt_mirna_scaled_data.T, 
    method="ward", 
    cmap="coolwarm", 
    row_cluster=True, 
    col_cluster=True,  # Cluster only samples
    col_colors=col_colors,  # Add phenotype bars
    figsize=(12, 8),
    vmin=-3,  # Set the lower bound of the color scale
    vmax=3,   # Set the upper bound of the color scale
    yticklabels=False,  # Remove row labels
    xticklabels=False,
    cbar_pos=None, #(0.9, 0.2, 0.03, 0.4)  # Move colorbar to the right
    dendrogram_ratio=(0.1, 0.1)
)

cbar = g.ax_heatmap.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)  # bigger tick text
cbar.set_label("Z-score", fontsize=16, fontweight='bold')  # set your label (not "ARI")
cbar.outline.set_linewidth(1.0)  # slightly thicker border

plt.suptitle("miRNA Hierarchical Clustering Heatmap - 50% Variance Filtered", 
             x=0.5, y=1.05, fontsize=14)
plt.show()

### Script Name: Copy Number (CN) Clustering

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script produces a clustered heatmap of the TARGET segement-level copy number information as processed by GISTIC2

**Requires**: TARGET sample phenotype matrix (use the methylation phenotype matrix as it overlapps completely with available CN information) and GISTIC2 derived TARGET segment-level copy number data matrix

In [ ]:
# load phenotype data - methylation phenotype dataset should suffice
meth_pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

seg_lvl_cn_raw = pd.read_excel('H:/Projects/Integrative_Clustering/Copy_Number/GISTIC2_output_95/all_lesions_95_CN.xlsx', index_col=0)
seg_lvl_cn = seg_lvl_cn_raw.iloc[:, 8:]

# transpose so that rows = samples, and columns = genes
cn_data_t = seg_lvl_cn.T

# Standardize the data (important for clustering) - This is actually the same thing as zscoring
scaler = StandardScaler()
scaled_data = scaler.fit_transform(cn_data_t) # normalizes across gene per smaple

scaled_data = pd.DataFrame(scaled_data, index=cn_data_t.index, columns=cn_data_t.columns) # convert back to a data frame

# remove values with zero variance
filt_cn_scaled_data = scaled_data.loc[:, scaled_data.var(axis=0) > 0]

# Define color mapping for "Chemoresponse" (binary: 1 = suboptimal, 2 = 0)
necrosis_palette = {'1': "#d8b365", '2': "#5ab4ac", " ": 'lightgray'}  # Adjust colors if needed

# Define color mapping for "Recurrence by 5yrs" (binary: 0 = No Recurrence, 1 = Recurrence)
recurrence_palette = {'0': "#2166ac", '1': "#b2182b", " ": 'lightgray'}  # Adjust colors if needed

# Create a DataFrame for row colors
col_colors = pd.DataFrame({
    "Chemoresponse": meth_pheno["ChemoResponse"].map(necrosis_palette),
    "Recurrence by 5yr": meth_pheno["Rec_by_5y"].map(recurrence_palette)
}, index=meth_pheno.index)

# Hierarchical clustering heatmap with phenotype annotations
sns.clustermap(
    filt_cn_scaled_data.T, 
    method="ward", 
    cmap="coolwarm", 
    row_cluster=True, 
    col_cluster=True,  # Cluster only samples
    col_colors=col_colors,  # Add phenotype bars
    figsize=(12, 8),
    vmin=-3,  # Set the lower bound of the color scale
    vmax=3,   # Set the upper bound of the color scale
    yticklabels=False,  # Remove row labels
    xticklabels=False,
    cbar_pos=None, #(0.9, 0.2, 0.03, 0.4)  # Move colorbar to the right
    dendrogram_ratio=(0.1, 0.1)
)

plt.suptitle("Seg-level Copy Number Hierarchical Clustering Heatmap", 
             x=0.5, y=1.05, fontsize=14)
plt.show()

### Script Name: Multi-Platform Cluster Concordance

**Authors**: Joshua Bowers

**Language**: Python

**Description**: This script defines concordance (ARI) between clustering methodlogies across all 4 platforms (methylation, miRNA, gene expression, and copy number)

**Requires**: TARGET sample phenotype matrix

In [ ]:
# Dependencies
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

pheno = pd.read_csv('H:/Projects/Methylation_Project/TARGET_Reproducibility/TARGET_Methylation.csv', index_col=0)

# Dictionary of cluster columns by number of clusters
# Optionally, you can add additionall numbers of clusters to this object
cluster_columns = {
    2: [
        # Methylation
        "eyeball_umap_clusters",

        # Expression
        "expr_90_mad_filt_hclust_2clust",

        # miRNA
        "mirna_50_variance_filt_hclust_2clust",

        # Copy Number
        "cn_hclust_seglvl_2c"
    ]
}

# Optionally, you can add additionall numbers of clusters to this object
# cluster_columns = {
#     2: [
#         # Methylation
#         "global_umap_clust",

#         # Expression
#         "expr_90_mad_filt_hclust_2clust",

#         # miRNA
#         "mirna_50_variance_filt_hclust_2clust",

#         # Copy Number
#         "cn_hclust_seglvl_2c"
#     ],

#     3: [
#         # Methylation
#         "global_umap_clust_3c", # This doesn't actually exist

#         # Expression
#         "expr_90_mad_filt_hclust_3clust",

#         # miRNA
#         "mirna_50_variance_filt_hclust_3clust",

#         # Copy Number
#         "cn_hclust_seglvl_3c"
#     ]
# }


##### Function to compute pairwise ARI matrix #####
def compute_ari_matrix(df, columns):
    n = len(columns)
    ari_matrix = pd.DataFrame(np.nan, index=columns, columns=columns)
    for i, col1 in enumerate(columns):
        for j, col2 in enumerate(columns):
            if i == j:
                ari_matrix.loc[col1, col2] = 1.0
            elif pd.isna(ari_matrix.loc[col1, col2]):
                valid_idx = df[[col1, col2]].dropna().index
                score = adjusted_rand_score(df.loc[valid_idx, col1], df.loc[valid_idx, col2]) if len(valid_idx) > 0 else np.nan
                ari_matrix.loc[col1, col2] = score
                ari_matrix.loc[col2, col1] = score
    return ari_matrix

##### Compute ARI matrices #####
ari_matrices = {}
for k in [2]: # This need to change if you're running the anlaysis of only 2 cluster solutions. Example: [2, 3, 4]
    cols = cluster_columns[k]
    valid_cols = [c for c in cols if c in pheno.columns]
    if not valid_cols:
        print(f" No valid columns found for {k}-cluster solution.")
        continue
    subset = pheno[valid_cols]
    ari_matrices[k] = compute_ari_matrix(subset, valid_cols)

# Unpack for easy access
ari_2 = ari_matrices.get(2)
#ari_3 = ari_matrices.get(3)
#ari_4 = ari_matrices.get(4)